# GMAI-Pulse — CoverMe Discovery Probe

**Purpose.** Pre-EDA read-only reconnaissance of the CoverMe hit-level table
(`csdo_prod_catalog.adobe_coverme_bronze.hit_data`, provisional) to answer the questions
that must be settled BEFORE the tailored CoverMe EDA / charts notebooks can be written:

1. **What report suites (rsids) live in this table?** Volume share, active windows,
   top hosts per rsid — so a business user can spot at a glance whether the table is
   single-suite (as the catalog name suggests) or carries bleed-over from other suites.
2. **What URLs bound the CoverMe scope?** Which URL column has the best coverage
   (`page_url`, `post_page_url`, `first_hit_page_url`, `visit_start_page_url`, `site_url`),
   what hosts exist (prod / uat / stage / aem-authoring / other), and what path prefixes
   cover the traffic.
3. **What is the schema and what's populated?** Full column census (dtype, non-blank %,
   approx-distinct) plus top-N raw values for every live eVar / prop / campaign / event.
4. **What is the time-series shape?** Volume trend, day-of-week × hour heatmap,
   weekly seasonality — the raw material an anomaly detector will consume.
5. **Three tiered scope recommendations** — tight / medium / broad — with hit-coverage
   percentages so business users can pick the scope filter for the follow-on notebooks.

**What this notebook is NOT.** It is not the production EDA notebook. It does not build
a synthesis spec, does not commit a scope filter, does not write to Delta. It emits
`===== BEGIN SHAREABLE: <id> =====` blocks that you paste back — from those, the
follow-on `coverme_eda.py` and `coverme_charts.py` get built with the scope pinned.

**Scope.** Runs UNFILTERED by default (all scope widgets empty). Every rsid, every URL,
every country present in the table is surfaced so nothing is silently dropped. Business
users can re-run with widgets populated to iteratively narrow.

**Known table quirks the probe now handles gracefully.**
1. `date_time` is a string column; a small fraction of rows carry `'0'` as the value
   (Adobe heartbeat / metadata rows). `resolve_date_expr` uses `try_to_timestamp` so
   those rows return NULL and coalesce through to `hit_time_gmt` / `hit_date` — they
   no longer take down every downstream section with a DateTimeException.
2. `csdo_prod_catalog.adobe_coverme_bronze.hit_data` is single-suite — the Adobe
   pipeline pins the report suite in feed config, so there is no `rsid` column. S3
   emits `single_suite: true` against a synthetic rsid label; S15 skips rsid filtering
   entirely and computes coverage from URL alone. That is the expected shape, not an
   error — do not adjust the `rsid_list` widget.

**Data visibility.** Every column profiles **raw and in full** — business dimensions
(eVars, props, events, pagenames, campaigns, referrers), URLs including any
`?query#fragment` suffix, AND direct device/network identifiers (visitor IDs, cookies,
IPs, `geo_zip`, `user_agent`). Emit-time scrubbing is off; top-N value lists print
verbatim for every profiled column. Authorized 2026-07-20 by the data owner, mirroring
the GWAM EDA policy. Downstream governance still applies at Bronze→Silver — identifiers
are HMAC-pseudonymized in the shipped pipeline per ADR-0007 §2.

**How to run.** Databricks → Workspace → Import → File → select this `.py` (it imports
as a notebook — the file is in Databricks "source" format). Attach to any cluster with
Unity Catalog access (DBR 13+ recommended). Run the **S0 config cell** once so the
widgets appear, then **Run All**. Expect S1/S2 in seconds, S3/S4 as the two full-table
scans (minutes each — no URL predicate, so partition-pruning is the only prune),
S5 as a third full-table scan (narrow projection), S6–S14 on the sample.

**What to paste back — priority order** if you can't share the whole exported `.ipynb`:

1. `synthesis_spec` (the master consolidation)
2. `scope_options` (three tiered filter profiles for business-user review)
3. `rsid_inventory` + `url_host_inventory` (the two headline discovery blocks)
4. `mobile_app_profile` (mobile-SDK traffic reading — drives mobile-only scope in the follow-on EDA)
5. `ts_daily`, `ts_events`, `ts_profiles` (time-series shape for anomaly-model design)
6. `event_decode`, `live_custom_dims`, `population_census` (metric-registry seeding)
7. `daily_volume`, `delta_meta`, `geo_language` (volume + freshness + geography)
8. `dim_candidates`, `dq_baseline`, `identity_evidence`, `schema_probe`, `run_manifest`

**If something goes wrong.** Sections are independent: a failure prints
`===== SKIPPED: <id> | <reason> =====` and the run continues — paste those back too.
If everything downstream is empty, S1 (schema_probe) is the first sanity check:
missing rsid / URL / timestamp columns are surfaced there. If the table doesn't
resolve at all, the `table_fqn` widget is wrong.

## S0 — Config, constants, helpers

In [0]:
import json
import re
import math
import hashlib
import datetime
import traceback

from pyspark.sql import functions as F
from pyspark.sql import Window

# ---------------------------------------------------------------- widgets ----
dbutils.widgets.text("table_fqn", "csdo_prod_catalog.adobe_coverme_bronze.hit_data", "1. Table (catalog.schema.table)")
dbutils.widgets.text("window_months", "13", "2. Deep-profiling window (months)")
dbutils.widgets.text("sample_fraction", "0.05", "3. Sample fraction for per-column stats")
dbutils.widgets.text("col_batch_size", "150", "4. Columns per aggregation batch")
dbutils.widgets.text("top_n", "20", "5. Top-N cap for value lists")
dbutils.widgets.text("hourly_days", "35", "6. Days for hourly profile")
dbutils.widgets.text("max_csv_lines", "450", "7. Max CSV lines per shareable block")
dbutils.widgets.text("top_events_k", "8", "8. Top-K events for daily series")
dbutils.widgets.text("cache_sample", "false", "9. Persist sample df (true/false)")
dbutils.widgets.text("rsid_list", "", "10. rsid list (comma-sep, EMPTY = discover all)")
dbutils.widgets.text("url_include", "", "11. URL include patterns (SQL LIKE, comma-sep, EMPTY = discover all)")
dbutils.widgets.text("url_exclude", "", "12. URL exclude patterns (SQL LIKE, comma-sep, EMPTY = off)")
dbutils.widgets.text("login_host_exclude", "", "13. Individual-login hosts to exclude (comma-sep, EMPTY = off)")
dbutils.widgets.text("pdf_labels_path", "", "14. Optional workspace path to the CoverMe PDFs (EMPTY = skip labels)")
dbutils.widgets.text("top_hosts_per_rsid_k", "10", "15. Top-K hosts per rsid in S3")
dbutils.widgets.text("top_hosts_k", "50", "16. Top-K hosts in S4b")
dbutils.widgets.text("top_prefixes_k", "100", "17. Top-K host+path-prefix rows in S4c")
dbutils.widgets.text("prefix_depth", "5", "18. Path segments in the S4c rollup prefix")

TABLE_FQN            = dbutils.widgets.get("table_fqn").strip()
WINDOW_MONTHS        = int(dbutils.widgets.get("window_months"))
SAMPLE_FRACTION      = float(dbutils.widgets.get("sample_fraction"))
COL_BATCH_SIZE       = int(dbutils.widgets.get("col_batch_size"))
TOP_N                = int(dbutils.widgets.get("top_n"))
HOURLY_DAYS          = int(dbutils.widgets.get("hourly_days"))
MAX_CSV_LINES        = int(dbutils.widgets.get("max_csv_lines"))
TOP_EVENTS_K         = int(dbutils.widgets.get("top_events_k"))
CACHE_SAMPLE         = dbutils.widgets.get("cache_sample").strip().lower() == "true"
TOP_HOSTS_PER_RSID_K = int(dbutils.widgets.get("top_hosts_per_rsid_k"))
TOP_HOSTS_K          = int(dbutils.widgets.get("top_hosts_k"))
TOP_PREFIXES_K       = int(dbutils.widgets.get("top_prefixes_k"))
PREFIX_DEPTH         = max(1, int(dbutils.widgets.get("prefix_depth")))
PDF_LABELS_PATH      = dbutils.widgets.get("pdf_labels_path").strip()


def _csv(widget):
    return [p.strip().lower() for p in dbutils.widgets.get(widget).split(",") if p.strip()]


RSID_LIST     = _csv("rsid_list")
URL_INCLUDE   = _csv("url_include")
URL_EXCLUDE   = _csv("url_exclude")
LOGIN_EXCLUDE = _csv("login_host_exclude")

# Scope-condition state: filled by S1 when it inspects the schema. If a scope widget is
# populated but the required column is missing, S1 emits a `missing_conditions` list so
# the run cannot silently profile the wrong population.
RSID_COL = None
URL_COL  = None
URL_COLS = None
DATE_EXPR = None
DATE_EXPR_DESC = None

# --------------------------------------------------- data-visibility policy ----
# All columns profile RAW — authorized by the data owner 2026-07-20, mirroring the
# GWAM EDA policy. Business dimensions, URLs (query strings included), and identifier
# columns (mcvisid, IPs, cookies, ZIPs, user_agent) all emit top-N value lists verbatim.

# ------------------------------------------------------------ emit helpers ----
RESULTS = {}   # section_id -> payload (drives S16 consolidation)
SKIPPED = {}   # section_id -> reason

# Cap any single string field so one runaway value can't blow past the Databricks
# per-cell stdout cap and hide the rest of the block.
MAX_EMIT_STR = 2000


def _scrub_str(s):
    if len(s) > MAX_EMIT_STR:
        s = s[:MAX_EMIT_STR] + "...<trunc>"
    return s


def _scrub(obj):
    """Walk a payload: truncate very long strings, round floats, pass everything else through."""
    if isinstance(obj, dict):
        return {(_scrub_str(k) if isinstance(k, str) else k): _scrub(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_scrub(v) for v in obj]
    if isinstance(obj, str):
        return _scrub_str(obj)
    if isinstance(obj, float):
        return round(obj, 4) if math.isfinite(obj) else None
    return obj


def emit(section_id, payload):
    """Print + register a shareable JSON block. Applies string truncation only."""
    payload = _scrub(payload)
    RESULTS[section_id] = payload
    body = json.dumps(payload, separators=(",", ":"), default=str)
    print(f"===== BEGIN SHAREABLE: {section_id} =====")
    if len(body) <= 48000:
        print(body)
    else:
        n_parts = math.ceil(len(body) / 40000)
        for i in range(n_parts):
            print(f"----- part {i+1} of {n_parts} (concatenate parts to reassemble) -----")
            print(body[i * 40000:(i + 1) * 40000])
    print(f"===== END SHAREABLE: {section_id} =====")


def run_section(section_id, fn):
    print(f"\n>>> running {section_id} ...")
    t0 = datetime.datetime.now()
    try:
        fn()
        print(f">>> {section_id} done in {(datetime.datetime.now() - t0).total_seconds():.0f}s")
    except Exception as e:
        reason = f"{type(e).__name__}: {str(e)[:300]}"
        SKIPPED[section_id] = reason
        print(f"===== SKIPPED: {section_id} | {reason} =====")
        traceback.print_exc()


# ------------------------------------------------------------ data helpers ----
def mask(v):
    """SHA1-truncated token; retained for parity with the GWAM notebook even though
    the current data-visibility policy prints raw values."""
    return "<masked:" + hashlib.sha1(str(v).encode("utf-8")).hexdigest()[:8] + ">"


def qcol(col_name):
    """F.col with backtick quoting — Adobe schemas can carry dotted column names
    (mobileappperformanceappid.*) that unquoted F.col parses as struct access."""
    return F.col("`" + col_name.replace("`", "``") + "`")


def nonblank(col_name):
    """Adobe feeds use empty strings, not NULLs."""
    c = qcol(col_name)
    return c.isNotNull() & (F.trim(c.cast("string")) != "")


def batched_agg(df, agg_exprs, batch_size):
    """Run many agg expressions in batches to dodge codegen limits.
    agg_exprs: list of (alias, Column). Returns {alias: value}."""
    out = {}
    for i in range(0, len(agg_exprs), batch_size):
        batch = agg_exprs[i:i + batch_size]
        exprs = [c.alias(a) for a, c in batch]
        try:
            row = df.agg(*exprs).collect()[0]
        except Exception:
            spark.conf.set("spark.sql.codegen.wholeStage", "false")
            try:
                row = df.agg(*exprs).collect()[0]
            finally:
                spark.conf.set("spark.sql.codegen.wholeStage", "true")
        out.update(row.asDict())
    return out


def resolve_date_expr(df):
    """Fallback chain for the canonical hit timestamp — never throws on malformed input.

    Composes a COALESCE (first non-NULL wins):
      1. date_time (typed timestamp/date) if already typed.
      2. try_to_timestamp(date_time) if it's a string — yields NULL on '0' /
         malformed values instead of DateTimeException (which breaks every
         downstream section that touches DATE_EXPR; this was the CoverMe run's
         root failure mode with the ~thousand '0'-valued heartbeat rows).
      3. from_unixtime(hit_time_gmt) as epoch-seconds fallback.
      4. hit_date (typed date) cast to midnight — last resort. Hour resolution
         is lost, but daily aggs and hit_date partition pruning still work.

    Coalesce (not preference) so a rare '0' in date_time doesn't silently drop
    the row — it falls through to hit_time_gmt / hit_date.
    """
    dtypes = dict(df.dtypes)
    have = set(dtypes.keys())
    parts, desc_parts = [], []
    if "date_time" in have:
        if dtypes["date_time"] in ("timestamp", "date"):
            parts.append(F.col("date_time").cast("timestamp"))
            desc_parts.append("date_time (typed)")
        else:
            parts.append(F.expr("try_to_timestamp(date_time)"))
            desc_parts.append("try_to_timestamp(date_time)")
    if "hit_time_gmt" in have:
        parts.append(F.from_unixtime(F.expr("try_cast(hit_time_gmt as bigint)")).cast("timestamp"))
        desc_parts.append("from_unixtime(hit_time_gmt)")
    if "hit_date" in have and dtypes["hit_date"] == "date":
        parts.append(F.col("hit_date").cast("timestamp"))
        desc_parts.append("hit_date (typed) @ midnight")
    if not parts:
        raise ValueError("No usable timestamp column (date_time / hit_time_gmt / hit_date) found")
    expr = F.coalesce(*parts) if len(parts) > 1 else parts[0]
    return expr, " COALESCE ".join(desc_parts)


def pick_col(df, *candidates):
    """First candidate column present in the schema, else None."""
    cols = set(df.columns)
    for c in candidates:
        if c in cols:
            return c
    return None


def _resolve_scope_cols(df):
    global RSID_COL, URL_COL, URL_COLS
    RSID_COL = pick_col(df, "rsid", "report_suite", "reportsuite", "reportsuiteid", "post_rsid")
    # D4 (from GWAM EDA S4c): page_url FIRST. post_page_url is blank 36-46% of the time
    # on GWAM report suites, so preferring it silently drops ~40% of rows. Verify for
    # CoverMe in S4a — if the pattern reverses, the recommendation will call it out.
    have = set(df.columns)
    URL_COLS = [c for c in ("page_url", "post_page_url", "first_hit_page_url",
                            "visit_start_page_url", "site_url") if c in have]
    URL_COL  = URL_COLS[0] if URL_COLS else None


def like_any(colexpr, patterns):
    """Null-safe OR of SQL LIKE patterns. Returns None when `patterns` is empty.

    Blank/NULL input yields False, never NULL, so `~like_any(...)` stays well-defined
    — a NULL there would silently drop the row instead of keeping it.
    """
    if not patterns:
        return None
    m = None
    for p in patterns:
        m = colexpr.like(p) if m is None else (m | colexpr.like(p))
    return F.coalesce(m, F.lit(False))


def url_expr(df):
    """D4 blank-guarded coalesce over the URL candidate columns, lowercased.

    Adobe writes empty strings, not NULLs, so a plain coalesce() returns "" from
    page_url and never falls through. Map blank -> NULL first, then coalesce, then
    land on "" so the NOT LIKE exclusions stay well-defined.
    """
    if URL_COLS is None:
        _resolve_scope_cols(df)
    if not URL_COLS:
        return None
    parts = []
    for c in URL_COLS:
        t = F.trim(F.col(c).cast("string"))
        parts.append(F.when(t != F.lit(""), t))
    return F.lower(F.coalesce(*parts, F.lit("")))


def scope_condition(df):
    """Scope selector honoring the widget-driven filter. Returns (Column|None, meta).

    Every widget defaults empty (DISCOVER mode) — cond is then None and downstream
    sections profile the full table. When a widget IS populated the condition tightens;
    missing schema columns get flagged in `meta.missing_conditions` so the run can't
    silently profile the wrong population.
    """
    if RSID_COL is None and URL_COLS is None:
        _resolve_scope_cols(df)
    conds, active, missing = [], [], []
    if RSID_LIST:
        if RSID_COL:
            conds.append(F.lower(F.trim(F.col(RSID_COL).cast("string"))).isin(RSID_LIST))
            active.append(f"rsid[{RSID_COL}] in {RSID_LIST}")
        else:
            missing.append("rsid (no report-suite column found)")
    u = url_expr(df)
    if u is None:
        if URL_INCLUDE or URL_EXCLUDE or LOGIN_EXCLUDE:
            missing.append("url (no page_url/post_page_url column found)")
    else:
        inc = like_any(u, URL_INCLUDE)
        if inc is not None:
            conds.append(inc)
            active.append(f"url[coalesce{tuple(URL_COLS)}] LIKE any {URL_INCLUDE}")
        exc = like_any(u, URL_EXCLUDE)
        if exc is not None:
            conds.append(~exc)
            active.append(f"url NOT LIKE any {URL_EXCLUDE}")
        lex = like_any(u, LOGIN_EXCLUDE)
        if lex is not None:
            conds.append(~lex)
            active.append(f"login hosts excluded: {LOGIN_EXCLUDE}")
    cond = None
    for c in conds:
        cond = c if cond is None else (cond & c)
    meta = {"rsid_col": RSID_COL, "url_col": URL_COL, "url_cols_coalesced": URL_COLS,
            "rsid_list": RSID_LIST or None, "url_include": URL_INCLUDE or None,
            "url_exclude": URL_EXCLUDE or None, "login_host_exclude": LOGIN_EXCLUDE or None,
            "active_conditions": active, "missing_conditions": missing,
            "scoped": cond is not None,
            "mode": "DISCOVER (no scope filter)" if cond is None else "SCOPED"}
    return cond, meta


# Globals populated by S1/S5/S6; ensure_frames() rebuilds them for re-runs.
DF = None
DF_CA = None
DF_W = None
DF_S = None
WINDOW_START = None
WINDOW_END = None
SAMPLE_ROWS = None

# PDF-derived labels (empty unless S0.5 runs successfully).
EVENT_LABELS = {}   # event_id (str) -> friendly_name
EVAR_LABELS  = {}   # variable_name (str, e.g. "evar15") -> friendly_name


def ensure_frames():
    """Make DF/DF_CA/DF_W/DF_S available even when a section is re-run standalone.
    DF = full table. DF_CA = optionally-scoped subset (identity when scope off).
    DF_W = last N months of DF_CA. DF_S = random sample of DF_W."""
    global DF, DF_CA, DF_W, DF_S, DATE_EXPR, DATE_EXPR_DESC, WINDOW_START, WINDOW_END, SAMPLE_ROWS
    if DF is None:
        DF = spark.table(TABLE_FQN)
    if DATE_EXPR is None:
        DATE_EXPR, DATE_EXPR_DESC = resolve_date_expr(DF)
    if DF_CA is None:
        cond, _ = scope_condition(DF)
        DF_CA = DF.filter(cond) if cond is not None else DF
    if DF_W is None:
        if WINDOW_END is None:
            dv = RESULTS.get("daily_volume", {})
            WINDOW_END = (datetime.date.fromisoformat(dv["date_max"])
                          if dv.get("date_max") else datetime.date.today())
        WINDOW_START = (WINDOW_END.replace(day=1) - datetime.timedelta(days=1)).replace(day=1)
        for _ in range(WINDOW_MONTHS - 1):
            WINDOW_START = (WINDOW_START - datetime.timedelta(days=1)).replace(day=1)
        DF_W = DF_CA.filter(F.to_date(DATE_EXPR) >= F.lit(WINDOW_START))
    if DF_S is None:
        DF_S = DF_W.sample(withReplacement=False, fraction=SAMPLE_FRACTION, seed=42)
        DF_S = DF_S.persist()      # always freeze: count() and per-column aggs must read identical rows
        SAMPLE_ROWS = DF_S.count() # materializes the persisted sample
    return DF, DF_W, DF_S


print(f"Config OK. table={TABLE_FQN}  window={WINDOW_MONTHS}mo  fraction={SAMPLE_FRACTION}  "
      f"batch={COL_BATCH_SIZE}  top_n={TOP_N}")
print(f"Scope filter: rsid={RSID_LIST or '(discover)'}  "
      f"include={URL_INCLUDE or '(discover)'}  "
      f"exclude={URL_EXCLUDE or '(off)'}  "
      f"login_hosts_excluded={len(LOGIN_EXCLUDE)}")
print(f"PDF labels: {PDF_LABELS_PATH or '(disabled)'}")

Config OK. table=csdo_prod_catalog.adobe_coverme_bronze.hit_data  window=13mo  fraction=0.05  batch=150  top_n=20
Scope filter: rsid=(discover)  include=(discover)  exclude=(off)  login_hosts_excluded=0
PDF labels: (disabled)


## S0.5 — Optional PDF label extraction
Runs only when `pdf_labels_path` widget is set to a workspace directory containing the
CoverMe reference PDFs (`Coverme_events(metrics).pdf`, `Variables_coverme_1-50.pdf`,
`Variables_coverme_50-104.pdf`, `Variables_coverme_200.pdf`). Requires `pypdf` on the
cluster — if the import fails, print the `%pip install pypdf` hint and skip. Never fatal.

In [0]:
def s05_pdf_labels():
    global EVENT_LABELS, EVAR_LABELS
    if not PDF_LABELS_PATH:
        emit("pdf_labels", {"enabled": False, "reason": "pdf_labels_path widget empty — skipped"})
        return

    try:
        import pypdf
    except ImportError:
        emit("pdf_labels", {"enabled": False,
                            "reason": "pypdf not installed — run `%pip install pypdf` in a "
                                      "separate cell above S0.5, then Restart Python and re-run"})
        return

    import os
    if not os.path.isdir(PDF_LABELS_PATH):
        emit("pdf_labels", {"enabled": False,
                            "reason": f"pdf_labels_path is not a directory: {PDF_LABELS_PATH}"})
        return

    def _read_pdf(path):
        with open(path, "rb") as fh:
            reader = pypdf.PdfReader(fh)
            return "\n".join((p.extract_text() or "") for p in reader.pages)

    files_found = {}
    for fname in ("Coverme_events(metrics).pdf", "Variables_coverme_1-50.pdf",
                  "Variables_coverme_50-104.pdf", "Variables_coverme_200.pdf"):
        full = os.path.join(PDF_LABELS_PATH, fname)
        files_found[fname] = os.path.isfile(full)

    # Separator chars between an ID and its friendly name in the PDFs: colon,
    # equals, period, whitespace, en/em dash, hyphen. Hyphen sits at the end of
    # the character class so it's unambiguously literal (no escape needed inside
    # []; escaping it is what triggers the "duplicate in character class" warning).
    _sep = r"[:=.\s\u2013\u2014-]+"

    # Events: look for lines with an event ID + a friendly name. Adobe events range
    # from 1..1000; standard IDs (1..20, 500..504) are Adobe-defined, custom starts >20.
    event_labels = {}
    events_pdf = os.path.join(PDF_LABELS_PATH, "Coverme_events(metrics).pdf")
    if os.path.isfile(events_pdf):
        text = _read_pdf(events_pdf)
        # Lenient patterns — match "event500 = clickmappage", "Event 500: Click Map Page",
        # "ev500 - clickmappage", "500  Click Map Page", etc.
        for m in re.finditer(r"\b(?:event\s*|ev)(\d{1,4})\s*" + _sep + r"([A-Za-z][^\n\r]{2,120})",
                             text, re.IGNORECASE):
            eid, name = m.group(1), m.group(2).strip()
            event_labels.setdefault(eid, name)
        # Column-format fallback: "500  Click Map Page  ..."
        for m in re.finditer(r"^\s*(\d{1,4})\s{2,}([A-Za-z][^\n\r]{2,120})$",
                             text, re.MULTILINE):
            eid, name = m.group(1), m.group(2).strip()
            event_labels.setdefault(eid, name)

    # eVars / props: pull "evar15 = province" style lines from every variables pdf.
    evar_labels = {}
    for fname in ("Variables_coverme_1-50.pdf", "Variables_coverme_50-104.pdf",
                  "Variables_coverme_200.pdf"):
        full = os.path.join(PDF_LABELS_PATH, fname)
        if not os.path.isfile(full):
            continue
        text = _read_pdf(full)
        for m in re.finditer(r"\b(evar|prop|hier)\s*(\d{1,3})\s*" + _sep + r"([A-Za-z][^\n\r]{2,140})",
                             text, re.IGNORECASE):
            kind, num, name = m.group(1).lower(), m.group(2), m.group(3).strip()
            evar_labels.setdefault(f"{kind}{num}", name)

    EVENT_LABELS = event_labels
    EVAR_LABELS  = evar_labels

    emit("pdf_labels", {
        "enabled": True,
        "path": PDF_LABELS_PATH,
        "files_found": files_found,
        "n_event_labels": len(event_labels),
        "n_evar_labels": len(evar_labels),
        "event_labels": dict(sorted(event_labels.items(), key=lambda kv: int(kv[0]))),
        "evar_labels": dict(sorted(evar_labels.items())),
    })


run_section("S0_5", s05_pdf_labels)


>>> running S0_5 ...
===== BEGIN SHAREABLE: pdf_labels =====
{"enabled":false,"reason":"pdf_labels_path widget empty \u2014 skipped"}
===== END SHAREABLE: pdf_labels =====
>>> S0_5 done in 0s


## S1 — Schema probe
Metadata only; runs in seconds. Confirms the table resolves, lists all columns with
dtypes, and resolves the scope-critical columns (rsid, URL, timestamp).

In [0]:
def s1_schema_probe():
    global DF
    resolves, err, all_cols, dtypes = False, None, [], {}
    try:
        DF = spark.table(TABLE_FQN)
        resolves = True
        all_cols = DF.columns
        dtypes = dict(DF.dtypes)
        _resolve_scope_cols(DF)
    except Exception as e:
        err = f"{type(e).__name__}: {str(e)[:200]}"

    date_col, date_desc = None, None
    if resolves:
        try:
            _, date_desc = resolve_date_expr(DF)
            date_col = date_desc
        except Exception as e:
            date_col = None
            date_desc = f"{type(e).__name__}: {str(e)[:200]}"

    # Group columns for readability — the SHAREABLE payload prints them individually,
    # but the counts up top give a business user a quick "how big is this schema?" answer.
    evar_cols = sorted(c for c in all_cols if re.match(r"^(post_)?evar\d+$", c))
    prop_cols = sorted(c for c in all_cols if re.match(r"^(post_)?prop\d+$", c))
    hier_cols = sorted(c for c in all_cols if re.match(r"^(post_)?hier\d+$", c))
    event_cols = sorted(c for c in all_cols if "event" in c.lower())
    url_cols  = sorted(c for c in all_cols if "url" in c.lower())
    id_cols   = sorted(c for c in all_cols if re.search(r"visid|cust_?visid|userid|user_?hash|cookies|ip\b|ipv6|mcvisid", c))
    geo_cols  = sorted(c for c in all_cols if c.startswith("geo_"))
    time_cols = sorted(c for c in all_cols if re.search(r"(date_?time|hit_?time|_?time_?gmt|load_?timestamp)", c))

    emit("schema_probe", {
        "configured_table": TABLE_FQN,
        "resolves": resolves,
        "resolve_error": err,
        "n_cols": len(all_cols),
        "date_column": date_col,
        "date_expr_desc": date_desc,
        "rsid_col": RSID_COL,
        "url_col_first_in_coalesce": URL_COL,
        "url_cols_present": URL_COLS,
        "column_groups": {
            "evar_count": len(evar_cols), "evar_cols_sample": evar_cols[:15],
            "prop_count": len(prop_cols), "prop_cols_sample": prop_cols[:15],
            "hier_count": len(hier_cols), "hier_cols_sample": hier_cols[:15],
            "event_cols": event_cols,
            "url_cols": url_cols,
            "identity_cols": id_cols,
            "geo_cols": geo_cols,
            "time_cols": time_cols,
        },
        "dtypes": dtypes,
        "note": "If resolves=false, set the table_fqn widget to the correct FQN and re-run.",
    })


run_section("S1", s1_schema_probe)


>>> running S1 ...
===== BEGIN SHAREABLE: schema_probe =====
{"configured_table":"csdo_prod_catalog.adobe_coverme_bronze.hit_data","resolves":true,"resolve_error":null,"n_cols":1180,"date_column":"try_to_timestamp(date_time) COALESCE from_unixtime(hit_time_gmt) COALESCE hit_date (typed) @ midnight","date_expr_desc":"try_to_timestamp(date_time) COALESCE from_unixtime(hit_time_gmt) COALESCE hit_date (typed) @ midnight","rsid_col":null,"url_col_first_in_coalesce":"page_url","url_cols_present":["page_url","post_page_url","first_hit_page_url","visit_start_page_url"],"column_groups":{"evar_count":500,"evar_cols_sample":["evar1","evar10","evar100","evar101","evar102","evar103","evar104","evar105","evar106","evar107","evar108","evar109","evar11","evar110","evar111"],"prop_count":150,"prop_cols_sample":["post_prop1","post_prop10","post_prop11","post_prop12","post_prop13","post_prop14","post_prop15","post_prop16","post_prop17","post_prop18","post_prop19","post_prop2","post_prop20","post_prop21"

## S2 — Delta metadata & load cadence
`DESCRIBE DETAIL` + `DESCRIBE HISTORY`: freshness / arrival-cadence evidence, zero data scan.

In [0]:
def s2_delta_meta():
    detail = spark.sql(f"DESCRIBE DETAIL {TABLE_FQN}").collect()[0].asDict()
    # Deliberately exclude 'location' and free-form properties (internal paths).
    detail_safe = {k: detail.get(k) for k in
                   ["format", "numFiles", "sizeInBytes", "partitionColumns",
                    "clusteringColumns", "createdAt", "lastModified"]}

    writes = {"available": False}
    try:
        hist = spark.sql(f"DESCRIBE HISTORY {TABLE_FQN} LIMIT 100").collect()
        write_ops = [h for h in hist if h.operation and
                     any(k in h.operation.upper() for k in ["WRITE", "MERGE", "UPDATE", "COPY", "REPLACE"])]
        ts = sorted([h.timestamp for h in write_ops])
        gaps_h = sorted((b - a).total_seconds() / 3600 for a, b in zip(ts, ts[1:]))
        recent = []
        for h in write_ops[:20]:
            om = h.operationMetrics or {}
            rows_written = om.get("numOutputRows") or om.get("numTargetRowsInserted")
            recent.append({"ts": str(h.timestamp), "op": h.operation, "rows": rows_written})
        ops_by_type = {}
        for h in hist:
            ops_by_type[h.operation] = ops_by_type.get(h.operation, 0) + 1
        writes = {
            "available": True,
            "n_history_rows": len(hist),
            "ops_by_type": ops_by_type,
            "n_write_ops": len(write_ops),
            "median_interarrival_hours": gaps_h[len(gaps_h) // 2] if gaps_h else None,
            "min_gap_hours": gaps_h[0] if gaps_h else None,
            "max_gap_hours": gaps_h[-1] if gaps_h else None,
            "recent_writes": recent,
        }
    except Exception as e:
        writes = {"available": False, "error": f"{type(e).__name__}: {str(e)[:200]}"}

    emit("delta_meta", {"detail": detail_safe, "writes": writes})


run_section("S2", s2_delta_meta)


>>> running S2 ...
===== BEGIN SHAREABLE: delta_meta =====
{"detail":{"format":"delta","numFiles":10625,"sizeInBytes":17117611893,"partitionColumns":["hit_date"],"clusteringColumns":[],"createdAt":"2023-09-21 18:19:28.715000","lastModified":"2026-07-22 07:10:04"},"writes":{"available":true,"n_history_rows":60,"ops_by_type":{"WRITE":60},"n_write_ops":60,"median_interarrival_hours":23.9989,"min_gap_hours":0.1364,"max_gap_hours":72.0728,"recent_writes":[{"ts":"2026-07-22 07:10:04","op":"WRITE","rows":"39916"},{"ts":"2026-07-21 07:10:48","op":"WRITE","rows":"37721"},{"ts":"2026-07-20 07:08:37","op":"WRITE","rows":"22686"},{"ts":"2026-07-19 07:10:30","op":"WRITE","rows":"24909"},{"ts":"2026-07-18 07:07:34","op":"WRITE","rows":"33331"},{"ts":"2026-07-17 07:12:48","op":"WRITE","rows":"36637"},{"ts":"2026-07-16 07:11:17","op":"WRITE","rows":"36205"},{"ts":"2026-07-15 07:11:06","op":"WRITE","rows":"37212"},{"ts":"2026-07-14 07:10:57","op":"WRITE","rows":"38027"},{"ts":"2026-07-13 07:08:34","op

## S3 — rsid inventory (business-user primary output #1)
No filter applied. Full-history scan projecting only rsid + date + URL columns
(partition-pruning is the only cost lever — this is the first of the three heavy scans).
For every rsid present, reports: hit count, % share, active days, first/last date,
monthly volume histogram (surfaces cutovers), and the top-K hosts under that rsid.
A business user seeing "rsid A = 92% coverme.com traffic, rsid B = 8% internal test"
can pick their scope from this block alone.

In [0]:
def s3_rsid_inventory():
    global DF, DATE_EXPR, DATE_EXPR_DESC
    if DF is None:
        DF = spark.table(TABLE_FQN)
    if DATE_EXPR is None:
        DATE_EXPR, DATE_EXPR_DESC = resolve_date_expr(DF)
    _resolve_scope_cols(DF)

    # Single-suite tables (e.g. csdo_prod_catalog.adobe_coverme_bronze) don't carry
    # an rsid column — Adobe pipelines pin the report suite in feed config, not
    # per-row. Emit the same payload shape against a synthetic rsid label so S15 /
    # S16 / S17 don't have to special-case the missing key.
    single_suite = RSID_COL is None
    if single_suite:
        rsid_norm = F.lit("<single_suite>")
    else:
        rsid_norm = F.lower(F.trim(F.col(RSID_COL).cast("string")))
    _u = url_expr(DF)
    host_expr = None
    if _u is not None:
        # Strip scheme + www + query/fragment, keep host up to the first '/'.
        _clean = F.regexp_replace(_u, r"^[a-z]+://", "")
        _clean = F.regexp_replace(_clean, r"^www\.", "")
        _hp = F.regexp_extract(_clean, r"^([^?#]*)", 1)
        host_expr = F.regexp_extract(_hp, r"^([^/]+)", 1)

    # ---- per-rsid rollup + monthly bucket in one scan ----
    proj_cols = [rsid_norm.alias("rsid"),
                 F.to_date(DATE_EXPR).alias("d"),
                 F.date_format(DATE_EXPR, "yyyy-MM").alias("ym")]
    if host_expr is not None:
        proj_cols.append(host_expr.alias("host"))

    proj = DF.select(*proj_cols).persist()
    total_rows = proj.count()

    per_rsid = (proj.groupBy("rsid").agg(
        F.count("*").alias("hits"),
        F.countDistinct("d").alias("active_days"),
        F.min("d").cast("string").alias("first_day"),
        F.max("d").cast("string").alias("last_day"),
    ).orderBy(F.desc("hits")).collect())

    rsid_list = []
    for r in per_rsid:
        rsid_list.append({
            "rsid": r["rsid"],
            "hits": r["hits"],
            "pct": round(100.0 * (r["hits"] or 0) / max(total_rows, 1), 3),
            "active_days": r["active_days"],
            "first_day": r["first_day"],
            "last_day": r["last_day"],
        })

    monthly = (proj.groupBy("rsid", "ym").agg(F.count("*").alias("hits"))
                   .orderBy("ym", "rsid").collect())
    monthly_hits = [{"month": r["ym"], "rsid": r["rsid"], "hits": r["hits"]} for r in monthly]

    # Top-K hosts per rsid, ranked within each rsid partition.
    top_hosts_per_rsid = {}
    if host_expr is not None:
        w = Window.partitionBy("rsid").orderBy(F.desc("hits"))
        ranked = (proj.filter(F.col("host") != F.lit(""))
                      .groupBy("rsid", "host").agg(F.count("*").alias("hits"))
                      .withColumn("rnk", F.row_number().over(w))
                      .filter(F.col("rnk") <= TOP_HOSTS_PER_RSID_K)
                      .orderBy("rsid", "rnk").collect())
        for r in ranked:
            top_hosts_per_rsid.setdefault(r["rsid"], []).append(
                {"host": r["host"], "hits": r["hits"]})

    proj.unpersist()

    emit("rsid_inventory", {
        "total_rows_all_history": total_rows,
        "rsid_col": RSID_COL,
        "single_suite": single_suite,
        "n_rsids": len(rsid_list),
        "per_rsid": rsid_list,
        "monthly_hits_by_rsid": monthly_hits,
        "top_hosts_per_rsid": top_hosts_per_rsid,
        "reading_note": (
            "Single-suite table (no rsid column in schema) — one implicit report "
            "suite covers every row. Monthly volume + top-hosts still apply; the "
            "follow-on scope filter should be URL-based, not rsid-based."
            if single_suite else
            "This is the primary business-user picker for rsid. If one rsid is "
            "≥95% of hits, it is almost certainly the CoverMe production suite. "
            "Cross-check its top hosts against your Adobe Analytics report suite "
            "definitions."
        ),
    })


run_section("S3", s3_rsid_inventory)


>>> running S3 ...
===== BEGIN SHAREABLE: rsid_inventory =====
{"total_rows_all_history":60102257,"rsid_col":null,"single_suite":true,"n_rsids":1,"per_rsid":[{"rsid":"<single_suite>","hits":60102257,"pct":100.0,"active_days":1210,"first_day":"2023-02-28","last_day":"2026-07-21"}],"monthly_hits_by_rsid":[{"month":null,"rsid":"<single_suite>","hits":10},{"month":"2023-02","rsid":"<single_suite>","hits":41717},{"month":"2023-03","rsid":"<single_suite>","hits":1099254},{"month":"2023-04","rsid":"<single_suite>","hits":894894},{"month":"2023-05","rsid":"<single_suite>","hits":930287},{"month":"2023-06","rsid":"<single_suite>","hits":943485},{"month":"2023-07","rsid":"<single_suite>","hits":923555},{"month":"2023-08","rsid":"<single_suite>","hits":916719},{"month":"2023-09","rsid":"<single_suite>","hits":808028},{"month":"2023-10","rsid":"<single_suite>","hits":1151478},{"month":"2023-11","rsid":"<single_suite>","hits":1338573},{"month":"2023-12","rsid":"<single_suite>","hits":1696348},{"mo

## S4 — URL / host inventory (business-user primary output #2)
Second of the three heavy scans. Three subsections:
  - **4a** — per-URL-column stats (blank %, distinct, top-100 coverage). Recommends
    the coalesce order for the follow-on notebook.
  - **4b** — top-N hosts with cumulative % coverage + prod/uat/stage/aem tag.
  - **4c** — top-N host + path-prefix (5 segments deep, IDs generalized) with
    cumulative % + language tag. Business users pick their URL scope from this table.

Query strings are stripped before grouping (host + path only) so distinct-URL blast
radius stays manageable and the rollup remains readable.

In [0]:
def _hp_expr(colexpr):
    """host + path from a raw URL: strip scheme, www., and query/fragment."""
    u = F.regexp_replace(F.lower(colexpr.cast("string")), r"^[a-z]+://", "")
    u = F.regexp_replace(u, r"^www\.", "")
    return F.regexp_extract(u, r"^([^?#]*)", 1)


def _host_from_hp(hp_col):
    return F.regexp_extract(hp_col, r"^([^/]+)", 1)


def _path_from_hp(hp_col):
    return F.when(hp_col.rlike(r"^[^/]+/"),
                  F.concat(F.lit("/"), F.regexp_extract(hp_col, r"^[^/]+/(.*)$", 1))
                  ).otherwise(F.lit("/"))


def _generalize_ids(path_col):
    """/member/12345 -> /member/{id}; a privacy control AND a readability lever
    (otherwise leaf IDs shatter the top-N rollup into singleton rows)."""
    p = path_col
    p = F.regexp_replace(p, r"/[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}(?=/|$)", "/{id}")
    p = F.regexp_replace(p, r"/[0-9a-f]{16,}(?=/|$)", "/{id}")
    p = F.regexp_replace(p, r"/\d{4,}(?=/|$)", "/{id}")
    return p


def _path_prefix(path_gen_col, depth):
    """First `depth` path segments joined with '/'."""
    parts = []
    for n in range(1, depth + 1):
        parts.append(F.regexp_extract(path_gen_col, r"^" + ("/[^/]*" * (n - 1)) + r"/([^/]*)", 1))
    return F.concat(
        F.lit("/"), parts[0],
        *[F.when(s != "", F.concat(F.lit("/"), s)).otherwise(F.lit("")) for s in parts[1:]],
    )


# Host-env classification regexes (used both in Spark expressions and driver-side Python).
_HOST_ENV_AEM   = r"adobeaemcloud\.com|author-|-author\."
_HOST_ENV_UAT   = r"(^|\.)uat[.\-]|(-|^)uat($|\.)"
_HOST_ENV_STAGE = r"stage|staging|preview|dev\-|^dev\."
_HOST_ENV_OTHER = r"localhost|\.local$|\.test$"


def _tag_host_env(host_col):
    """prod / uat / stage / aem_authoring / other (Spark expression form)."""
    return (F.when(host_col.rlike(_HOST_ENV_AEM), "aem_authoring")
             .when(host_col.rlike(_HOST_ENV_UAT), "uat")
             .when(host_col.rlike(_HOST_ENV_STAGE), "stage")
             .when(host_col.rlike(_HOST_ENV_OTHER), "other")
             .otherwise("prod"))


def _env_tag_py(host):
    """Driver-side twin of _tag_host_env for tagging rows collected to Python."""
    h = host or ""
    if re.search(_HOST_ENV_AEM, h):
        return "aem_authoring"
    if re.search(_HOST_ENV_UAT, h):
        return "uat"
    if re.search(_HOST_ENV_STAGE, h):
        return "stage"
    if re.search(_HOST_ENV_OTHER, h):
        return "other"
    return "prod"


def _tag_lang(path_col, host_col=None):
    """Language tag by path token (GWAM style) + host domain (CoverMe style).

    CoverMe separates languages by top-level domain (pourmeproteger.com is the FR
    twin of coverme.com; manuvie / assurance-manuvie for FR); GWAM uses `/ca/en/`
    vs `/ca/fr/` path segments. Cover both so a single tagger works across suites.
    """
    path_tag = (F.when(path_col.rlike(r"(^|/)fr(/|$)")
                        | path_col.rlike(r"regimes-collectif|retraite|particuliers|assurance-|blogue"), "fr")
                 .when(path_col.rlike(r"(^|/)en(/|$)")
                        | path_col.rlike(r"group-retirement|group-plans"), "en"))
    if host_col is None:
        return path_tag.otherwise("unknown")
    host_tag = (F.when(host_col.rlike(r"pourmeproteger|manuvie|assurance-manuvie"), "fr")
                 .when(host_col.rlike(r"coverme\.com|insttrip\.manulife\.com|-en\."), "en"))
    return F.coalesce(path_tag, host_tag, F.lit("unknown"))


def s4_url_host_inventory():
    global DF, DATE_EXPR
    if DF is None:
        DF = spark.table(TABLE_FQN)
    if DATE_EXPR is None:
        DATE_EXPR, _ = resolve_date_expr(DF)
    _resolve_scope_cols(DF)

    if not URL_COLS:
        emit("url_host_inventory", {"error": "no URL columns found in the schema — check schema_probe"})
        return

    # ---------------- 4a: per-URL-column stats (one scan across all columns) ----
    # For each candidate URL column: blank rate, approx-distinct on host+path, and the
    # share of rows where THIS column matches the D4 coalesce (i.e. would contribute).
    coal = url_expr(DF)   # blank-guarded lowercase coalesce
    coal_hp = F.regexp_extract(F.regexp_replace(coal, r"^[a-z]+://", ""), r"^([^?#]*)", 1)

    exprs_4a = [F.count("*").alias("rows")]
    for c in URL_COLS:
        exprs_4a += [
            F.sum(F.when(nonblank(c), 1).otherwise(0)).alias(c + "_nb"),
            F.approx_count_distinct(_hp_expr(F.col(c))).alias(c + "_dist"),
            F.sum(F.when(_hp_expr(F.col(c)) == coal_hp, 1).otherwise(0)).alias(c + "_matches_coal"),
        ]
    row4a = DF.agg(*exprs_4a).collect()[0].asDict()
    total4a = row4a["rows"] or 0

    per_url_column = {}
    for c in URL_COLS:
        nb = row4a[c + "_nb"] or 0
        per_url_column[c] = {
            "blank_pct": round(100.0 * (total4a - nb) / max(total4a, 1), 3),
            "nonblank_hits": nb,
            "approx_distinct_host_path": row4a[c + "_dist"],
            "rows_matching_coalesce": row4a[c + "_matches_coal"],
        }

    # Recommendation: order by lowest blank_pct — that column contributes most to the coalesce.
    coalesce_recommended = sorted(URL_COLS, key=lambda c: per_url_column[c]["blank_pct"])
    coalesce_reasoning = "; ".join(
        f"{c} blank={per_url_column[c]['blank_pct']}%" for c in coalesce_recommended
    )

    # ---------------- 4b: top hosts across the whole table ----
    hp = _hp_expr(coal)
    host = _host_from_hp(hp)

    with_host = DF.select(host.alias("host")).filter(F.col("host") != F.lit("")).persist()
    total_host_rows = with_host.count()

    host_rollup = (with_host.groupBy("host").agg(F.count("*").alias("hits"))
                            .orderBy(F.desc("hits")).limit(TOP_HOSTS_K).collect())
    with_host.unpersist()

    top_hosts, cum = [], 0
    for r in host_rollup:
        cum += r["hits"]
        top_hosts.append({
            "host": r["host"],
            "hits": r["hits"],
            "pct": round(100.0 * r["hits"] / max(total_host_rows, 1), 3),
            "cumulative_pct": round(100.0 * cum / max(total_host_rows, 1), 3),
            # env tag applied host-by-host on the driver, so we don't need a second Spark job.
            "env": _env_tag_py(r["host"]),
        })

    # ---------------- 4c: top host + path-prefix (5 segments, IDs -> {id}) ----
    path = _path_from_hp(hp)
    path_gen = _generalize_ids(path)
    prefix = _path_prefix(path_gen, PREFIX_DEPTH)

    prefix_df = (DF.select(host.alias("host"),
                           prefix.alias("path_prefix"),
                           _tag_lang(path_gen, host).alias("lang"))
                   .filter(F.col("host") != F.lit(""))).persist()
    total_prefix_rows = prefix_df.count()

    prefix_rollup = (prefix_df.groupBy("host", "path_prefix", "lang")
                              .agg(F.count("*").alias("hits"))
                              .orderBy(F.desc("hits"))
                              .limit(TOP_PREFIXES_K).collect())
    prefix_df.unpersist()

    top_prefixes, cum = [], 0
    for r in prefix_rollup:
        cum += r["hits"]
        h = r["host"] or ""
        top_prefixes.append({
            "host": h,
            "path_prefix": r["path_prefix"],
            "lang": r["lang"],
            "hits": r["hits"],
            "pct": round(100.0 * r["hits"] / max(total_prefix_rows, 1), 3),
            "cumulative_pct": round(100.0 * cum / max(total_prefix_rows, 1), 3),
            "env": _env_tag_py(h),
        })

    emit("url_host_inventory", {
        "total_rows_with_host": total_host_rows,
        "s4a_per_url_column": per_url_column,
        "s4a_coalesce_recommended": coalesce_recommended,
        "s4a_coalesce_reasoning": coalesce_reasoning,
        "s4a_note": ("The recommended coalesce order minimizes blank %. If a column has "
                     "<0.1% blanks, it should lead the coalesce even if it isn't the "
                     "pipeline default. GWAM S4c found post_page_url was 37-46% blank vs "
                     "page_url <=0.013% — check for the same pattern here."),
        "s4b_top_hosts": top_hosts,
        "s4b_top_hosts_k": TOP_HOSTS_K,
        "s4c_top_host_path_prefixes": top_prefixes,
        "s4c_top_prefixes_k": TOP_PREFIXES_K,
        "s4c_prefix_depth": PREFIX_DEPTH,
        "reading_note": ("For each row in s4b/s4c, `cumulative_pct` tells you 'if I stop "
                         "here, what share of hits do I cover?'. `env` flags dev/staging/"
                         "authoring traffic that a production filter should exclude."),
    })


run_section("S4", s4_url_host_inventory)


>>> running S4 ...
===== BEGIN SHAREABLE: url_host_inventory =====
{"total_rows_with_host":60101107,"s4a_per_url_column":{"page_url":{"blank_pct":0.0,"nonblank_hits":60101967,"approx_distinct_host_path":4058,"rows_matching_coalesce":3891710},"post_page_url":{"blank_pct":58.896,"nonblank_hits":24704295,"approx_distinct_host_path":3509,"rows_matching_coalesce":1210987},"first_hit_page_url":{"blank_pct":16.872,"nonblank_hits":49961808,"approx_distinct_host_path":1528,"rows_matching_coalesce":748817},"visit_start_page_url":{"blank_pct":16.737,"nonblank_hits":50043088,"approx_distinct_host_path":2257,"rows_matching_coalesce":1230384}},"s4a_coalesce_recommended":["page_url","visit_start_page_url","first_hit_page_url","post_page_url"],"s4a_coalesce_reasoning":"page_url blank=0.0%; visit_start_page_url blank=16.737%; first_hit_page_url blank=16.872%; post_page_url blank=58.896%","s4a_note":"The recommended coalesce order minimizes blank %. If a column has <0.1% blanks, it should lead the coal

## S5 — Full-range daily volume
Third and last full-table scan (narrow projection: date + optional scope indicator).
Daily row counts over ALL history → history depth, missing days, day-of-week profile,
monthly totals, biggest day-over-day jumps. Both a total-table series and a
scope-matched series (when scope widgets are set) are reported side by side.

In [0]:
DAILY_TOTAL_ROWS = []   # [(date, total_count)] — whole-table series
DAILY_SCOPED_ROWS = []  # [(date, scope_count)] — scope-matched series (== total when scope off)


def s5_daily_volume():
    global DF, DATE_EXPR, DATE_EXPR_DESC, WINDOW_END, DAILY_TOTAL_ROWS, DAILY_SCOPED_ROWS
    if DF is None:
        DF = spark.table(TABLE_FQN)
    DATE_EXPR, DATE_EXPR_DESC = resolve_date_expr(DF)
    _resolve_scope_cols(DF)
    cond, scope_meta = scope_condition(DF)
    scoped_expr = F.when(cond, 1).otherwise(0) if cond is not None else F.lit(1)

    rows = (DF.select(F.to_date(DATE_EXPR).alias("d"), scoped_expr.alias("s"))
              .groupBy("d")
              .agg(F.count("*").alias("total"), F.sum("s").alias("scoped"))
              .orderBy("d").collect())
    null_dates = sum(r["total"] for r in rows if r["d"] is None)
    per_day = [(r["d"], r["total"], r["scoped"] or 0) for r in rows if r["d"] is not None]
    DAILY_TOTAL_ROWS  = [(d, t) for d, t, _ in per_day]
    DAILY_SCOPED_ROWS = [(d, s) for d, _, s in per_day]

    if not per_day:
        emit("daily_volume", {"error": "no non-null dates", "date_expr": DATE_EXPR_DESC,
                              "scope": scope_meta})
        return

    total_all = sum(t for _, t, _ in per_day)
    total_scoped = sum(s for _, _, s in per_day)

    dmin_all, dmax_all = per_day[0][0], per_day[-1][0]
    WINDOW_END = dmax_all

    # If a scope filter is active, some days may have 0 scoped hits; report the scoped
    # active range separately.
    scoped_active = [(d, s) for d, _, s in per_day if s > 0]
    dmin_scoped = scoped_active[0][0] if scoped_active else None
    dmax_scoped = scoped_active[-1][0] if scoped_active else None

    missing = []
    d = dmin_all
    while d <= dmax_all:
        if all(pd_d != d for pd_d, _, _ in per_day):
            missing.append(str(d))
        d += datetime.timedelta(days=1)

    # DoW means on the scoped series (Mon..Sun)
    dow_sum, dow_n = [0] * 7, [0] * 7
    for d, _, s in per_day:
        dow_sum[d.weekday()] += s
        dow_n[d.weekday()] += 1
    dow_mean = [round(dow_sum[i] / dow_n[i]) if dow_n[i] else None for i in range(7)]

    # monthly totals for both series — surface seasonality (RRSP-like or otherwise)
    monthly_total, monthly_scoped = {}, {}
    for d, t, s in per_day:
        ym = d.strftime("%Y-%m")
        monthly_total[ym] = monthly_total.get(ym, 0) + t
        monthly_scoped[ym] = monthly_scoped.get(ym, 0) + s

    # top |log-ratio| day-over-day jumps on the scoped series
    jumps = []
    for (d0, _, s0), (d1, _, s1) in zip(per_day, per_day[1:]):
        if s0 > 0 and s1 > 0:
            jumps.append((abs(math.log(s1 / s0)), str(d1), s0, s1))
    jumps.sort(reverse=True)
    top_jumps = [{"date": j[1], "prev": j[2], "curr": j[3],
                  "ratio": round(j[3] / j[2], 3)} for j in jumps[:5]]

    # CSV: date,total_hits,scoped_hits — full daily if it fits, else last WINDOW_MONTHS
    csv_header = "date,total_hits,scoped_hits"
    if len(per_day) <= MAX_CSV_LINES:
        csv_daily = [f"{d},{t},{s}" for d, t, s in per_day]
        csv_note = "full history, daily"
    else:
        cutoff = (dmax_all.replace(day=1) - datetime.timedelta(days=1)).replace(day=1)
        for _ in range(WINDOW_MONTHS - 1):
            cutoff = (cutoff - datetime.timedelta(days=1)).replace(day=1)
        csv_daily = [f"{d},{t},{s}" for d, t, s in per_day if d >= cutoff][:MAX_CSV_LINES]
        csv_note = f"daily since {cutoff}; older history in monthly_totals"

    emit("daily_volume", {
        "date_expr": DATE_EXPR_DESC,
        "scope": scope_meta,
        "total_rows_all_history": total_all,
        "total_rows_scoped": total_scoped,
        "scoped_share_pct": round(100.0 * total_scoped / max(total_all, 1), 3),
        "null_date_rows": null_dates,
        "date_min": str(dmin_all),
        "date_max": str(dmax_all),
        "n_days_present": len(per_day),
        "n_days_missing": len(missing),
        "missing_days": missing[:50],
        "scoped_date_min": str(dmin_scoped) if dmin_scoped else None,
        "scoped_date_max": str(dmax_scoped) if dmax_scoped else None,
        "dow_mean_scoped_hits_mon_to_sun": dow_mean,
        "monthly_totals_all": monthly_total,
        "monthly_totals_scoped": monthly_scoped,
        "top5_day_over_day_jumps_scoped": top_jumps,
        "csv_note": csv_note,
        "csv_header": csv_header,
        "csv": csv_daily,
    })


run_section("S5", s5_daily_volume)


>>> running S5 ...
===== BEGIN SHAREABLE: daily_volume =====
{"date_expr":"try_to_timestamp(date_time) COALESCE from_unixtime(hit_time_gmt) COALESCE hit_date (typed) @ midnight","scope":{"rsid_col":null,"url_col":"page_url","url_cols_coalesced":["page_url","post_page_url","first_hit_page_url","visit_start_page_url"],"rsid_list":null,"url_include":null,"url_exclude":null,"login_host_exclude":null,"active_conditions":[],"missing_conditions":[],"scoped":false,"mode":"DISCOVER (no scope filter)"},"total_rows_all_history":60102247,"total_rows_scoped":60102247,"scoped_share_pct":100.0,"null_date_rows":10,"date_min":"2023-02-28","date_max":"2026-07-21","n_days_present":1210,"n_days_missing":30,"missing_days":["2023-04-09","2023-04-10","2023-04-11","2023-04-12","2023-12-19","2023-12-20","2023-12-21","2024-01-07","2024-12-09","2025-08-05","2025-08-06","2025-08-07","2025-09-08","2025-09-09","2025-09-15","2025-11-10","2025-11-12","2025-11-18","2025-12-02","2025-12-03","2025-12-05","2025-12-06","

In [0]:
# chart for your own inspection (not part of the shareable output)
if DAILY_TOTAL_ROWS:
    display(spark.createDataFrame(
        [(str(d), t, s) for (d, t), (_, s) in zip(DAILY_TOTAL_ROWS, DAILY_SCOPED_ROWS)],
        ["date", "total_hits", "scoped_hits"]))

date,total_hits,scoped_hits
2023-02-28,41717,41717
2023-03-01,40379,40379
2023-03-02,36831,36831
2023-03-03,34969,34969
2023-03-04,24204,24204
2023-03-05,25569,25569
2023-03-06,47682,47682
2023-03-07,42455,42455
2023-03-08,41200,41200
2023-03-09,39769,39769


## S6 — Profiling window + sample frames
Builds `df_w` (last N months of DF_CA — DF_CA is DF unless scope widgets are set)
and `df_s` (random sample). Cross-checks the window count against S5.

In [0]:
def s6_frames():
    global DF_CA, DF_W, DF_S, SAMPLE_ROWS
    DF_CA = None; DF_W = None; DF_S = None  # force rebuild with S5's WINDOW_END
    ensure_frames()
    window_rows_scoped = DF_W.count()
    s5_window_sum = sum(s for d, s in DAILY_SCOPED_ROWS if d >= WINDOW_START) if DAILY_SCOPED_ROWS else None
    _, scope_meta = scope_condition(DF)
    emit("window_frame", {
        "window_start": str(WINDOW_START), "window_end": str(WINDOW_END),
        "window_rows_scoped": window_rows_scoped,
        "s5_crosscheck_sum_scoped": s5_window_sum,
        "crosscheck_ok": (s5_window_sum == window_rows_scoped) if s5_window_sum is not None else None,
        "sample_fraction": SAMPLE_FRACTION,
        "sample_rows": SAMPLE_ROWS,
        "sample_cached": CACHE_SAMPLE,
        "scope": scope_meta,
    })


run_section("S6", s6_frames)


>>> running S6 ...
===== BEGIN SHAREABLE: window_frame =====
{"window_start":"2025-06-01","window_end":"2026-07-21","window_rows_scoped":17886270,"s5_crosscheck_sum_scoped":17886270,"crosscheck_ok":true,"sample_fraction":0.05,"sample_rows":893283,"sample_cached":false,"scope":{"rsid_col":null,"url_col":"page_url","url_cols_coalesced":["page_url","post_page_url","first_hit_page_url","visit_start_page_url"],"rsid_list":null,"url_include":null,"url_exclude":null,"login_host_exclude":null,"active_conditions":[],"missing_conditions":[],"scoped":false,"mode":"DISCOVER (no scope filter)"}}
===== END SHAREABLE: window_frame =====
>>> S6 done in 12s


## S7 — Population census
Which columns are actually populated? Batched non-blank counts on the sample,
then approx-distinct only for live columns. Never prints values.

In [0]:
CENSUS = {}   # col -> {"dtype":..., "pop_pct":..., "apx_distinct":...}; reused by S8/S12/S14
CORE_MIN_PCT = 99.0   # "core" = reliably populated; usable as a stable series


def s7_population_census():
    global CENSUS
    ensure_frames()
    all_cols = DF_S.columns
    dtypes = dict(DF_S.dtypes)

    pop_exprs = [(c, F.sum(F.when(nonblank(c), 1).otherwise(0))) for c in all_cols]
    pop_counts = batched_agg(DF_S, pop_exprs, COL_BATCH_SIZE)

    n = max(SAMPLE_ROWS, 1)
    populated = {c: cnt for c, cnt in pop_counts.items() if (cnt or 0) / n >= 0.001}
    sparse    = [c for c, cnt in pop_counts.items() if 0 < (cnt or 0) / n < 0.001]
    dead      = [c for c, cnt in pop_counts.items() if not cnt]

    dist_exprs = [(c, F.approx_count_distinct(qcol(c))) for c in populated]
    distincts = batched_agg(DF_S, dist_exprs, COL_BATCH_SIZE) if dist_exprs else {}

    CENSUS = {c: {"dtype": dtypes.get(c), "pop_pct": round(100.0 * pop_counts[c] / n, 3),
                  "apx_distinct": distincts.get(c)} for c in populated}
    core = {c for c in CENSUS if CENSUS[c]["pop_pct"] >= CORE_MIN_PCT}

    ranked = sorted(CENSUS.items(), key=lambda kv: -kv[1]["pop_pct"])
    emit("population_census", {
        "basis": "sample", "sample_rows": SAMPLE_ROWS,
        "n_total_cols": len(all_cols),
        "n_populated": len(populated), "n_sparse": len(sparse), "n_dead": len(dead),
        "n_core": len(core), "core_min_pct": CORE_MIN_PCT,
        "populated": [{"col": c, **v} for c, v in ranked[:150]],
        "populated_names_beyond_top150": [c for c, _ in ranked[150:]],
        "sparse_cols": sparse[:40],
        "evar_live": sorted(c for c in populated if re.match(r"post_evar\d+$|evar\d+$", c)),
        "prop_live": sorted(c for c in populated if re.match(r"post_prop\d+$|prop\d+$", c)),
        "hier_live": sorted(c for c in populated if re.match(r"post_hier\d+$|hier\d+$", c)),
        "evar_core": sorted(c for c in core if re.match(r"post_evar\d+$|evar\d+$", c)),
        "prop_core": sorted(c for c in core if re.match(r"post_prop\d+$|prop\d+$", c)),
    })


run_section("S7", s7_population_census)


>>> running S7 ...
===== BEGIN SHAREABLE: population_census =====
{"basis":"sample","sample_rows":893283,"n_total_cols":1180,"n_populated":304,"n_sparse":225,"n_dead":651,"n_core":95,"core_min_pct":99.0,"populated":[{"col":"browser","dtype":"string","pop_pct":100.0,"apx_distinct":797},{"col":"browser_height","dtype":"string","pop_pct":100.0,"apx_distinct":1700},{"col":"browser_width","dtype":"string","pop_pct":100.0,"apx_distinct":2537},{"col":"click_action_type","dtype":"string","pop_pct":100.0,"apx_distinct":3},{"col":"click_context_type","dtype":"string","pop_pct":100.0,"apx_distinct":2},{"col":"click_sourceid","dtype":"string","pop_pct":100.0,"apx_distinct":1},{"col":"code_ver","dtype":"string","pop_pct":100.0,"apx_distinct":11},{"col":"color","dtype":"string","pop_pct":100.0,"apx_distinct":4},{"col":"connection_type","dtype":"string","pop_pct":100.0,"apx_distinct":2},{"col":"cookies","dtype":"string","pop_pct":100.0,"apx_distinct":3},{"col":"country","dtype":"string","pop_pct":10

## S8 — Live eVars / props / campaign
Shape + raw top-value distributions for the live custom dimensions. Enriched with
PDF-derived labels when S0.5 loaded them.

In [0]:
def _evar_key(col_name):
    """Normalize post_evar15 / evar15 -> evar15 so PDF-derived EVAR_LABELS keys match."""
    m = re.match(r"^(?:post_)?(evar|prop|hier)(\d+)$", col_name)
    if m:
        return m.group(1) + m.group(2)
    return col_name


def s8_live_custom_dims():
    ensure_frames()
    live_all = [c for c in CENSUS
                if re.match(r"post_evar\d+$|evar\d+$|post_prop\d+$|prop\d+$|"
                            r"post_hier\d+$|hier\d+$|^post_campaign$|^campaign$", c)]
    live = sorted(live_all, key=lambda c: -CENSUS[c]["pop_pct"])[:30]
    if not live:
        emit("live_custom_dims", {"error": "no live eVar/prop/hier/campaign columns (run S7 first)"})
        return
    n_core = sum(1 for c in live_all if CENSUS[c]["pop_pct"] >= CORE_MIN_PCT)

    out = []
    for c in live:
        stats = DF_S.filter(nonblank(c)).agg(
            F.expr(f"percentile_approx(length(`{c}`), 0.5)").alias("len_p50"),
            F.max(F.length(c)).alias("len_max"),
            F.avg(F.length(c)).alias("len_avg"),
            F.avg(F.when(F.col(c).cast("string").startswith("http"), 1.0).otherwise(0.0)).alias("url_frac"),
        ).collect()[0]
        top = (DF_S.filter(nonblank(c)).groupBy(c).count()
                   .orderBy(F.desc("count")).limit(TOP_N).collect())
        pop_rows = max(SAMPLE_ROWS * CENSUS[c]["pop_pct"] / 100.0, 1)
        out.append({
            "col": c,
            "label": EVAR_LABELS.get(_evar_key(c)),
            "pop_pct": CENSUS[c]["pop_pct"],
            "apx_distinct": CENSUS[c]["apx_distinct"],
            "len": {"p50": stats["len_p50"], "avg": stats["len_avg"], "max": stats["len_max"]},
            "looks_like_url": (stats["url_frac"] or 0) > 0.5,
            "free_text": (stats["len_avg"] or 0) > 80,
            "top": [{"v": str(r[c]), "len": len(str(r[c])),
                     "pct": round(100.0 * r["count"] / pop_rows, 2)} for r in top],
            "mode": "raw",
        })
    emit("live_custom_dims", {"basis": "sample", "n_live": len(live_all), "n_core": n_core,
                              "labels_loaded": bool(EVAR_LABELS), "dims": out})


run_section("S8", s8_live_custom_dims)


>>> running S8 ...
===== BEGIN SHAREABLE: live_custom_dims =====
{"basis":"sample","n_live":151,"n_core":0,"labels_loaded":false,"dims":[{"col":"post_evar52","label":null,"pop_pct":89.012,"apx_distinct":234,"len":{"p50":26,"avg":26.0443,"max":88},"looks_like_url":false,"free_text":false,"top":[{"v":"covme:","len":6,"pct":23.75},{"v":"covme:health-insurance","len":22,"pct":11.36},{"v":"covme:home","len":10,"pct":4.93},{"v":"covme:travel-insurance:visitors-to-canada","len":41,"pct":4.35},{"v":"covme:hd:quote:customer-details","len":31,"pct":3.78},{"v":"covme:life-insurance","len":20,"pct":3.28},{"v":"covme:hd:quote:select-your-plan","len":31,"pct":3.03},{"v":"covme:health-insurance:follow-me","len":32,"pct":2.9},{"v":"covme:health-insurance:flexcare","len":31,"pct":2.83},{"v":"covme:hd:quote:review-quote","len":27,"pct":2.68},{"v":"covme:content:affinity-travel:en-ca:coverme:booking","len":51,"pct":2.54},{"v":"covme:travel-insurance:travelling-canadians","len":43,"pct":2.27},{"v":"covme

## S9 — post_event_list decode
Event-ID frequency + events-per-hit distribution. IDs enriched with PDF labels when
S0.5 loaded them; Adobe standard IDs (1=purchase, 2=product_view, 10-14=cart,
20=campaign_view, 500-504=clickmap) labeled inline as a fallback.

In [0]:
ADOBE_STD_EVENTS = {
    "1": "purchase", "2": "product_view", "10": "cart_open", "11": "checkout",
    "12": "cart_add", "13": "cart_remove", "14": "cart_view",
    "20": "campaign_view", "500": "clickmap_page", "501": "clickmap_link",
    "502": "clickmap_region", "503": "clickmap_link_by_region", "504": "target_session_id",
}
TOP_EVENT_IDS = []   # filled here; used by S10


def s9_event_decode():
    global TOP_EVENT_IDS
    ensure_frames()
    ev_col = pick_col(DF_S, "post_event_list", "event_list")
    if not ev_col:
        emit("event_decode", {"error": "no post_event_list/event_list column"})
        return

    events_arr = F.filter(
        F.transform(F.split(F.col(ev_col), ","), lambda x: F.trim(x)),
        lambda x: x != "",
    )
    base = DF_S.select(F.when(nonblank(ev_col), events_arr)
                        .otherwise(F.array().cast("array<string>")).alias("ev"))

    per_hit = base.agg(
        F.count("*").alias("hits"),
        F.sum(F.when(F.size("ev") > 0, 1).otherwise(0)).alias("hits_with_events"),
        F.expr("percentile_approx(size(ev), array(0.5, 0.95))").alias("pcts"),
        F.max(F.size("ev")).alias("max_events"),
    ).collect()[0]

    with_ev = base.filter(F.size("ev") > 0)
    # instances (every occurrence) vs hit-presence (array_distinct)
    inst = (with_ev.select(F.explode("ev").alias("e"))
            .select(F.split("e", "=")[0].alias("event_id"),
                    F.expr("try_cast(try_element_at(split(e, '='), 2) as double)").alias("val"))
            .groupBy("event_id")
            .agg(F.count("*").alias("instances"),
                 F.sum(F.when(F.col("val").isNotNull(), 1).otherwise(0)).alias("with_value"),
                 F.avg("val").alias("val_mean"), F.max("val").alias("val_max")))
    pres = (with_ev.select(F.explode(F.array_distinct(
                F.transform("ev", lambda x: F.split(x, "=")[0]))).alias("event_id"))
            .groupBy("event_id").agg(F.count("*").alias("hits_with")))
    freq = (inst.join(pres, "event_id", "outer")
                .orderBy(F.desc("hits_with")).limit(60).collect())

    hits = max(per_hit["hits"], 1)
    event_freq = []
    for r in freq:
        eid = r["event_id"]
        # PDF label first, then Adobe standard, then unknown fallback.
        label = (EVENT_LABELS.get(eid)
                 or ADOBE_STD_EVENTS.get(eid)
                 or "unknown — resolve via CoverMe events PDF or Adobe data dictionary")
        event_freq.append({
            "event_id": eid,
            "label": label,
            "label_source": ("pdf" if eid in EVENT_LABELS
                             else "adobe_std" if eid in ADOBE_STD_EVENTS
                             else "unknown"),
            "hits_with_pct": round(100.0 * (r["hits_with"] or 0) / hits, 3),
            "instances": r["instances"],
            "has_value_pct": (round(100.0 * (r["with_value"] or 0) / r["instances"], 2)
                              if r["instances"] else None),
            "val_mean": r["val_mean"], "val_max": r["val_max"],
        })
    TOP_EVENT_IDS = [e["event_id"] for e in event_freq[:TOP_EVENTS_K]]

    emit("event_decode", {
        "basis": "sample", "source_col": ev_col, "sample_hits": per_hit["hits"],
        "pct_hits_with_events": round(100.0 * per_hit["hits_with_events"] / hits, 2),
        "events_per_hit_p50_p95": list(per_hit["pcts"]) if per_hit["pcts"] else None,
        "events_per_hit_max": per_hit["max_events"],
        "labels_loaded": bool(EVENT_LABELS),
        "event_freq": event_freq,
    })


run_section("S9", s9_event_decode)


>>> running S9 ...
===== BEGIN SHAREABLE: event_decode =====
{"basis":"sample","source_col":"post_event_list","sample_hits":893283,"pct_hits_with_events":93.11,"events_per_hit_p50_p95":[14,30],"events_per_hit_max":47,"labels_loaded":false,"event_freq":[{"event_id":"164","label":"unknown \u2014 resolve via CoverMe events PDF or Adobe data dictionary","label_source":"unknown","hits_with_pct":70.937,"instances":633669,"has_value_pct":0.0,"val_mean":2.0,"val_max":2.0},{"event_id":"151","label":"unknown \u2014 resolve via CoverMe events PDF or Adobe data dictionary","label_source":"unknown","hits_with_pct":69.145,"instances":617664,"has_value_pct":0.0,"val_mean":2.0,"val_max":2.0},{"event_id":"103","label":"unknown \u2014 resolve via CoverMe events PDF or Adobe data dictionary","label_source":"unknown","hits_with_pct":59.151,"instances":528382,"has_value_pct":0.01,"val_mean":2.0,"val_max":2.0},{"event_id":"10017","label":"unknown \u2014 resolve via CoverMe events PDF or Adobe data dictiona

## S10 — Time-series pack
Exact daily hits/visits/visitors/clean-hits over the window + per-day series for the
top-K event IDs + 7×24 day-of-week × hour profile. This is what the anomaly model
design consumes.

In [0]:
TS_DAILY_PDF = None   # kept for the chart cell below


def s10_time_series():
    global TS_DAILY_PDF
    ensure_frames()
    vis_hi = pick_col(DF_W, "post_visid_high", "visid_high")
    vis_lo = pick_col(DF_W, "post_visid_low", "visid_low")
    visit_num = pick_col(DF_W, "visit_num")
    excl = pick_col(DF_W, "exclude_hit")

    aggs = [F.count("*").alias("hits")]
    if vis_hi and vis_lo and visit_num:
        aggs.append(F.approx_count_distinct(
            F.concat_ws(":", vis_hi, vis_lo, visit_num)).alias("visits"))
        aggs.append(F.approx_count_distinct(
            F.concat_ws(":", vis_hi, vis_lo)).alias("visitors"))
    if excl:
        aggs.append(F.sum(F.when(F.coalesce(F.expr(f"try_cast({excl} as int)"), F.lit(0)) == 0, 1)
                          .otherwise(0)).alias("clean_hits"))

    daily = (DF_W.groupBy(F.to_date(DATE_EXPR).alias("d")).agg(*aggs).orderBy("d").collect())
    cols = ["hits", "visits", "visitors", "clean_hits"]
    series = {c: [] for c in cols}
    dates = []
    for r in daily:
        if r["d"] is None:
            continue
        dates.append(r["d"])
        rd = r.asDict()
        for c in cols:
            series[c].append(rd.get(c))

    csv_daily = [",".join([str(d)] + [str(series[c][i]) if series[c][i] is not None else ""
                                      for c in cols]) for i, d in enumerate(dates)]

    # ---- per-day series for top-K event IDs (exact hit-presence counts) ----
    csv_events, ev_cols = [], []
    ev_col = pick_col(DF_W, "post_event_list", "event_list")
    if ev_col and TOP_EVENT_IDS:
        ev_daily = (DF_W.filter(nonblank(ev_col))
                    .select(F.to_date(DATE_EXPR).alias("d"),
                            F.explode(F.array_distinct(F.transform(
                                F.filter(F.transform(F.split(F.col(ev_col), ","), lambda x: F.trim(x)),
                                         lambda x: x != ""),
                                lambda x: F.split(x, "=")[0]))).alias("event_id"))
                    .filter(F.col("event_id").isin(TOP_EVENT_IDS))
                    .groupBy("d", "event_id").count().collect())
        by_date = {}
        for r in ev_daily:
            if r["d"] is not None:
                by_date.setdefault(r["d"], {})[r["event_id"]] = r["count"]
        ev_cols = TOP_EVENT_IDS
        csv_events = [",".join([str(d)] + [str(by_date.get(d, {}).get(e, 0)) for e in ev_cols])
                      for d in dates]

    # ---- hourly 7x24 profile over the last HOURLY_DAYS days ----
    hour_matrix = None
    if dates:
        h_start = dates[-1] - datetime.timedelta(days=HOURLY_DAYS)
        hourly = (DF_W.filter(F.to_date(DATE_EXPR) >= F.lit(h_start))
                  .select(F.to_date(DATE_EXPR).alias("d"), F.hour(DATE_EXPR).alias("h"))
                  .groupBy("d", "h").count()
                  .groupBy(F.dayofweek("d").alias("dow"), "h")
                  .agg(F.avg("count").alias("mean_hits")).collect())
        # dayofweek: 1=Sunday..7=Saturday -> reorder rows to Mon..Sun
        mat = [[0] * 24 for _ in range(7)]
        for r in hourly:
            mat[(r["dow"] + 5) % 7][r["h"]] = round(r["mean_hits"], 1)
        hour_matrix = mat

    # ---- driver-side stats ----
    profiles = {}
    try:
        import pandas as pd
        s = pd.Series(series["hits"], index=pd.to_datetime([str(d) for d in dates]))
        overall = s.mean()
        dow_idx = (s.groupby(s.index.dayofweek).mean() / overall).round(3)
        roll = s.rolling(7, center=True).median()
        shift_scores = (roll / roll.shift(7)).apply(
            lambda x: abs(math.log(x)) if x and x > 0 else 0)
        top_shifts = shift_scores.nlargest(5)
        profiles = {
            "cv": round(float(s.std() / overall), 4) if overall else None,
            "autocorr_lag7": round(float(s.autocorr(7)), 4) if len(s) > 14 else None,
            "autocorr_lag28": round(float(s.autocorr(28)), 4) if len(s) > 56 else None,
            "dow_index_mon_to_sun": [float(dow_idx.get(i, float("nan"))) for i in range(7)],
            "level_shift_candidates": [
                {"date": str(d.date()), "abs_log_ratio_wow": round(float(v), 3)}
                for d, v in top_shifts.items() if v > math.log(1.3)],
        }
        TS_DAILY_PDF = s.reset_index()
    except Exception as e:
        profiles = {"error": f"pandas stats failed: {type(e).__name__}: {str(e)[:150]}"}

    emit("ts_daily", {
        "basis": "exact_window",
        "csv_header": "date," + ",".join(cols),
        "csv": csv_daily[-MAX_CSV_LINES:],
        "visits_visitors_note": "approx_count_distinct (~5% rsd)" if vis_hi else "visid columns missing",
    })
    if csv_events:
        emit("ts_events", {
            "basis": "exact_window (hits containing event, not instances)",
            "csv_header": "date," + ",".join("ev" + e for e in ev_cols),
            "csv": csv_events[-MAX_CSV_LINES:],
            "event_labels": {e: (EVENT_LABELS.get(e) or ADOBE_STD_EVENTS.get(e) or "unknown")
                             for e in ev_cols},
        })
    emit("ts_profiles", {
        "hour_matrix_rows_mon_to_sun_cols_0_23h": hour_matrix,
        "hourly_days": HOURLY_DAYS,
        **profiles,
    })


run_section("S10", s10_time_series)


>>> running S10 ...
===== BEGIN SHAREABLE: ts_daily =====
{"basis":"exact_window","csv_header":"date,hits,visits,visitors,clean_hits","csv":["2025-06-01,33726,7459,5367,29916","2025-06-02,50159,9325,6697,45695","2025-06-03,49937,8921,7251,45323","2025-06-04,47928,8732,6297,43440","2025-06-05,47481,8540,6848,42724","2025-06-06,40468,8491,5437,36112","2025-06-07,31359,6884,4855,27566","2025-06-08,33946,7680,5486,30079","2025-06-09,53157,9662,7626,48503","2025-06-10,49362,7981,6752,44722","2025-06-11,48448,8903,6435,44126","2025-06-12,48229,9163,6274,43799","2025-06-13,39950,8383,5706,35873","2025-06-14,31183,6549,5292,27465","2025-06-15,31681,7092,5470,27941","2025-06-16,48350,9387,6814,44153","2025-06-17,45567,9479,6468,40729","2025-06-18,48373,8842,7025,43818","2025-06-19,46183,9754,6452,41879","2025-06-20,41554,8173,5797,37096","2025-06-21,31648,7102,4818,27724","2025-06-22,32041,7354,5404,28133","2025-06-23,45071,8981,6356,40481","2025-06-24,44666,9049,6524,39941","2025-06-25,44625,

In [0]:
# chart for your own inspection
if TS_DAILY_PDF is not None:
    display(spark.createDataFrame(TS_DAILY_PDF.astype(str)))

index,0
2025-06-01,33726
2025-06-02,50159
2025-06-03,49937
2025-06-04,47928
2025-06-05,47481
2025-06-06,40468
2025-06-07,31359
2025-06-08,33946
2025-06-09,53157
2025-06-10,49362


## S11 — Geography + language distribution
First-class discovery section since the CoverMe sample shows non-Canada traffic
(SGP / USA visitors via corporate proxies). Country / region / city / language
breakdown; feeds the `geo_country_scope` recommendation in S15.

In [0]:
# Adobe language codes seen in the wild (from the GWAM EDA notebook):
# 45 = English, 39 = French. Other codes surface here as-is; add labels as they're identified.
_LANG_LABELS = {"45": "English", "39": "French"}


def s11_geo_language():
    ensure_frames()
    out = {}

    if pick_col(DF_S, "geo_country"):
        gc = (DF_S.filter(nonblank("geo_country"))
                  .groupBy("geo_country").count()
                  .orderBy(F.desc("count")).limit(TOP_N).collect())
        n = max(SAMPLE_ROWS, 1)
        cum, top_c = 0, []
        for r in gc:
            cum += r["count"]
            top_c.append({"country": r["geo_country"],
                          "hits": r["count"],
                          "pct": round(100.0 * r["count"] / n, 3),
                          "cumulative_pct": round(100.0 * cum / n, 3)})
        out["top_countries"] = top_c
        out["is_canada_only"] = (len(top_c) == 1 and (top_c[0]["country"] or "").lower() == "can")

    if pick_col(DF_S, "geo_region"):
        gr = (DF_S.filter(nonblank("geo_region"))
                  .groupBy("geo_region").count()
                  .orderBy(F.desc("count")).limit(TOP_N).collect())
        out["top_regions"] = [{"region": r["geo_region"], "hits": r["count"]} for r in gr]

    if pick_col(DF_S, "geo_city"):
        gy = (DF_S.filter(nonblank("geo_city"))
                  .groupBy("geo_city").count()
                  .orderBy(F.desc("count")).limit(TOP_N).collect())
        out["top_cities"] = [{"city": r["geo_city"], "hits": r["count"]} for r in gy]

    if pick_col(DF_S, "language"):
        lg = (DF_S.filter(nonblank("language"))
                  .groupBy(F.col("language").cast("string").alias("language")).count()
                  .orderBy(F.desc("count")).limit(TOP_N).collect())
        n = max(SAMPLE_ROWS, 1)
        out["languages"] = [{"code": r["language"],
                             "label": _LANG_LABELS.get(str(r["language"])),
                             "hits": r["count"],
                             "pct": round(100.0 * r["count"] / n, 3)}
                            for r in lg]

    if not out:
        out["error"] = "no geo_* or language columns in the sample"

    emit("geo_language", {"basis": "sample", **out})


run_section("S11", s11_geo_language)


>>> running S11 ...
===== BEGIN SHAREABLE: geo_language =====
{"basis":"sample","top_countries":[{"country":"can","hits":753713,"pct":84.376,"cumulative_pct":84.376},{"country":"usa","hits":109552,"pct":12.264,"cumulative_pct":96.64},{"country":"fra","hits":3581,"pct":0.401,"cumulative_pct":97.04},{"country":"ind","hits":3049,"pct":0.341,"cumulative_pct":97.382},{"country":"gbr","hits":1863,"pct":0.209,"cumulative_pct":97.59},{"country":"phl","hits":1733,"pct":0.194,"cumulative_pct":97.784},{"country":"hkg","hits":1324,"pct":0.148,"cumulative_pct":97.933},{"country":"mex","hits":1110,"pct":0.124,"cumulative_pct":98.057},{"country":"aus","hits":1095,"pct":0.123,"cumulative_pct":98.179},{"country":"irl","hits":960,"pct":0.107,"cumulative_pct":98.287},{"country":"chn","hits":664,"pct":0.074,"cumulative_pct":98.361},{"country":"sgp","hits":658,"pct":0.074,"cumulative_pct":98.435},{"country":"jpn","hits":634,"pct":0.071,"cumulative_pct":98.506},{"country":"deu","hits":601,"pct":0.067,"cumu

## S11b — Mobile-app traffic profile
First-class discovery for mobile-SDK traffic. The CoverMe schema carries 100+ `mobile*`
columns (`mobileappid`, `mobiledevice`, `mobileosversion`, `mobileaction`, `mobilelaunches`,
`mobilemessage*`, `mobileplace*`, `mobilebeacon*`, `mobilelaunch*`, `mobilecrash*`,
`mobileltv*`, `mobileacquisition*` and the `post_*` shadows). If mobile-app traffic is
>0.1 % of hits, the follow-on `coverme_eda.py` should carve out a dedicated mobile scope;
if <0.1 %, a web-only scope is safe. Runs on the sample so cost is trivial. Must sit
BEFORE S15 so `scope_options.mobile_traffic` picks up a real reading.

In [0]:
def s11b_mobile_profile():
    """First-class discovery for mobile-SDK traffic (Adobe mobile SDKs).

    Answers: is mobile-app traffic present? If yes, what apps / devices / OS
    versions dominate, and which Adobe mobile subsystems are instrumented
    (message vs place vs beacon vs launch vs crash vs LTV)?
    """
    ensure_frames()

    # Full schema view of mobile columns (not just live ones — CENSUS is post-population).
    mobile_cols_schema = sorted(c for c in DF_S.columns if c.startswith("mobile"))
    mobile_cols_live = sorted(c for c in mobile_cols_schema
                              if c in CENSUS and CENSUS[c]["pop_pct"] >= 0.1)

    # Hit-level "is mobile-app" signal: mobileappid populated OR hit_source matches Adobe
    # mobile-SDK patterns. Adobe mobile-SDK hit_source values include 'mobile', 'sdk_*',
    # 'app_measurement'; web-side values are 'javascript', 'server_call', or blank.
    signals = []
    if "mobileappid" in DF_S.columns:
        signals.append(nonblank("mobileappid"))
    if "hit_source" in DF_S.columns:
        signals.append(F.lower(F.col("hit_source").cast("string"))
                        .rlike(r"^(mobile|sdk|app_measurement)"))
    if signals:
        is_mobile = signals[0]
        for s in signals[1:]:
            is_mobile = is_mobile | s
    else:
        is_mobile = F.lit(False)

    mobile_hits = DF_S.filter(is_mobile).count()
    mobile_hit_share_pct = round(100.0 * mobile_hits / max(SAMPLE_ROWS, 1), 3)

    def _top(col, k=TOP_N):
        if col not in DF_S.columns:
            return None
        pop = (DF_S.filter(nonblank(col)).groupBy(col)
                   .count().orderBy(F.desc("count")).limit(k).collect())
        if not pop:
            return None
        n_nonblank = max(sum(r["count"] for r in pop), 1)
        return [{"v": str(r[col]), "hits": r["count"],
                 "pct_of_nonblank": round(100.0 * r["count"] / n_nonblank, 3)}
                for r in pop]

    top_apps = _top("mobileappid")
    top_devices = _top("mobiledevice")
    top_os = _top("mobileosversion")
    top_hit_sources = _top("hit_source")
    top_mobile_actions = _top("mobileaction")

    # Adobe mobile-SDK subsystem buckets — how many columns in each are populated?
    _buckets = {
        "app_metadata": (r"^mobile(app(id|store)?|install|launch|upgrade|day|hour|"
                         r"osenvironment|osversion|resolution|device|crashrate|"
                         r"prevsession|ltv|carrier)"),
        "message":      r"^mobilemessage",
        "place":        r"^mobileplace",
        "beacon":       r"^mobilebeacon",
        "campaign":     r"^mobile(campaign|acquisition|deeplink|relaunch)",
        "performance":  r"^mobileappperformance",
        "crash":        r"^mobile(crash|appperformancecrash)",
        "push":         r"^mobile(pushoptin|pushpayload|messagepush)",
    }
    category_counts = {}
    for cat, pat in _buckets.items():
        cols = [c for c in mobile_cols_schema if re.match(pat, c)]
        live = [c for c in cols if c in CENSUS and CENSUS[c]["pop_pct"] >= 0.1]
        category_counts[cat] = {"schema_cols": len(cols), "populated_cols": len(live)}

    mobile_present = mobile_hit_share_pct >= 0.1 or bool(top_apps)

    recommendation = (
        "Mobile-app traffic detected \u2014 recommend a `mobile_only` scope tier in "
        "coverme_eda.py (URL filter alone won't catch it; key off mobileappid + "
        "hit_source)."
        if mobile_present else
        "Mobile-app traffic is <0.1% of sampled hits \u2014 a web-only scope is safe. "
        "The 100+ mobile* columns are present in the schema but effectively dead."
    )

    emit("mobile_app_profile", {
        "basis": "sample", "sample_rows": SAMPLE_ROWS,
        "mobile_present": mobile_present,
        "mobile_hit_share_pct": mobile_hit_share_pct,
        "n_mobile_cols_in_schema": len(mobile_cols_schema),
        "n_mobile_cols_live_pop_gte_0p1pct": len(mobile_cols_live),
        "adobe_mobile_categories": category_counts,
        "top_mobile_app_ids": top_apps,
        "top_mobile_devices": top_devices,
        "top_mobile_os_versions": top_os,
        "top_hit_sources": top_hit_sources,
        "top_mobile_actions": top_mobile_actions,
        "recommendation": recommendation,
    })


run_section("S11b", s11b_mobile_profile)


>>> running S11b ...
===== BEGIN SHAREABLE: mobile_app_profile =====
{"basis":"sample","sample_rows":893283,"mobile_present":false,"mobile_hit_share_pct":0.0,"n_mobile_cols_in_schema":100,"n_mobile_cols_live_pop_gte_0p1pct":1,"adobe_mobile_categories":{"app_metadata":{"schema_cols":54,"populated_cols":0},"message":{"schema_cols":12,"populated_cols":0},"place":{"schema_cols":6,"populated_cols":0},"beacon":{"schema_cols":4,"populated_cols":0},"campaign":{"schema_cols":14,"populated_cols":0},"performance":{"schema_cols":8,"populated_cols":0},"crash":{"schema_cols":5,"populated_cols":0},"push":{"schema_cols":5,"populated_cols":0}},"top_mobile_app_ids":null,"top_mobile_devices":null,"top_mobile_os_versions":null,"top_hit_sources":[{"v":"1","hits":893277,"pct_of_nonblank":99.999},{"v":"2","hits":6,"pct_of_nonblank":0.001}],"top_mobile_actions":null,"recommendation":"Mobile-app traffic is <0.1% of sampled hits \u2014 a web-only scope is safe. The 100+ mobile* columns are present in the schem

## S12 — Dimension candidates
Cardinality + top values of candidate slicing dimensions. Everything prints raw,
including any URL query strings.

In [0]:
def s12_dimensions():
    ensure_frames()
    dim_candidates = [c for c in [
        "pagename", "post_pagename", "page_url", "post_page_url", "referrer",
        "ref_domain", "ref_type", "browser", "os", "connection_type", "hit_source",
        "exclude_hit", "duplicate_purchase", "new_visit", "post_page_event",
        "va_closer_id", "va_finder_id", "channel", "post_channel",
        "mobileappid", "mobiledevice", "mobileosversion", "mobileaction",
    ] if c in set(DF_S.columns) and c in CENSUS]

    out = []
    for c in dim_candidates:
        top = (DF_S.filter(nonblank(c)).groupBy(c).count()
                   .orderBy(F.desc("count")).limit(TOP_N).collect())
        pop_rows = max(SAMPLE_ROWS * CENSUS[c]["pop_pct"] / 100.0, 1)
        top_vals = [{"v": str(r[c]), "pct": round(100.0 * r["count"] / pop_rows, 2)}
                    for r in top]
        out.append({"dim": c, "mode": "raw",
                    "coverage_pct": CENSUS[c]["pop_pct"],
                    "apx_distinct": CENSUS[c]["apx_distinct"],
                    "top": top_vals})
    emit("dim_candidates", {"basis": "sample", "dims": out})


run_section("S12", s12_dimensions)


>>> running S12 ...
===== BEGIN SHAREABLE: dim_candidates =====
{"basis":"sample","dims":[{"dim":"pagename","mode":"raw","coverage_pct":77.799,"apx_distinct":233,"top":[{"v":"covme:","pct":22.85},{"v":"covme:health-insurance","pct":10.89},{"v":"covme:home","pct":4.77},{"v":"covme:hd:quote:customer-details","pct":4.35},{"v":"covme:travel-insurance:visitors-to-canada","pct":4.25},{"v":"covme:hd:quote:select-your-plan","pct":3.97},{"v":"covme:hd:quote:review-quote","pct":3.48},{"v":"covme:health-insurance:follow-me","pct":2.78},{"v":"covme:health-insurance:flexcare","pct":2.72},{"v":"covme:life-insurance","pct":2.61},{"v":"covme:hd:null","pct":2.49},{"v":"covme:content:affinity-travel:en-ca:coverme:booking","pct":2.05},{"v":"covme:travel-insurance:travelling-canadians","pct":1.97},{"v":"covme:hd:plan-selector:applicant-information","pct":1.81},{"v":"covme:hd:quote:plan-details","pct":1.55},{"v":"covme:health-insurance:compare-plans","pct":0.99},{"v":"covme:life:quote:eligibility-question

## S13 — Data-quality baseline
Bot-filter distributions (exclude_hit × hit_source), clock skew, duplicate rate
(exact, one recent day), late-arrival evidence if a load-timestamp column exists.

In [0]:
def s13_dq_baseline():
    ensure_frames()
    key_cols = [c for c in ["date_time", "hit_time_gmt", "visit_num", "visit_page_num",
                            "post_event_list", "pagename", "page_url", "exclude_hit",
                            "hit_source"] if c in set(DF_S.columns)]
    key_nulls = {c: round(100.0 - CENSUS.get(c, {}).get("pop_pct", 0.0), 3) if c in CENSUS
                 else 100.0 for c in key_cols}

    # exclude_hit x hit_source (bot filtering rules)
    dist = []
    if pick_col(DF_S, "exclude_hit") and pick_col(DF_S, "hit_source"):
        dist = [{"exclude_hit": str(r["exclude_hit"]), "hit_source": str(r["hit_source"]),
                 "pct": round(100.0 * r["count"] / max(SAMPLE_ROWS, 1), 3)}
                for r in (DF_S.groupBy("exclude_hit", "hit_source").count()
                              .orderBy(F.desc("count")).limit(20).collect())]

    # clock skew: date_time (local) vs hit_time_gmt (epoch) -> tz offset + stragglers
    skew = None
    if pick_col(DF_S, "hit_time_gmt"):
        skew_row = (DF_S.filter(nonblank("hit_time_gmt"))
                    .select((F.unix_timestamp(DATE_EXPR)
                             - F.col("hit_time_gmt").cast("long")).alias("skew_s"))
                    .agg(F.expr("percentile_approx(skew_s, array(0.05, 0.5, 0.95))")
                         .alias("p")).collect()[0])
        skew = {"p5_p50_p95_seconds": list(skew_row["p"]) if skew_row["p"] else None,
                "note": "constant offset = timezone of date_time; spread = clock skew"}

    # duplicates: exact on the most recent complete day
    dup = None
    vis_hi = pick_col(DF_S, "post_visid_high", "visid_high")
    vis_lo = pick_col(DF_S, "post_visid_low", "visid_low")
    seq = pick_col(DF_S, "visit_page_num", "hit_time_gmt")
    days = [d for d, s in DAILY_SCOPED_ROWS if s > 0]
    if days and vis_hi and vis_lo and pick_col(DF_S, "visit_num") and seq:
        check_day = days[-2] if len(days) >= 2 else days[-1]
        day_df = DF_CA.filter(F.to_date(DATE_EXPR) == F.lit(check_day))
        total = day_df.count()
        distinct = day_df.select(vis_hi, vis_lo, "visit_num", seq).distinct().count()
        dup = {"day": str(check_day), "rows": total, "distinct_keys": distinct,
               "dup_pct": round(100.0 * (total - distinct) / max(total, 1), 4),
               "key": f"{vis_hi},{vis_lo},visit_num,{seq}", "basis": "exact_one_day"}

    # late-arrival: look for a load/ingest timestamp column
    load_cols = [c for c in DF_S.columns
                 if re.search(r"((load|ingest|etl|insert|_created|processed).*(ts|time|date))|_ts$",
                              c, re.IGNORECASE)]
    late = {"load_timestamp_cols_found": load_cols[:10]}
    if load_cols:
        lc = load_cols[0]
        try:
            late_row = (DF_S.filter(nonblank(lc))
                        .select(F.datediff(F.to_date(qcol(lc).cast("timestamp")),
                                           F.to_date(DATE_EXPR)).alias("lag_days"))
                        .agg(F.expr("percentile_approx(lag_days, array(0.5, 0.95, 0.99))")
                             .alias("p")).collect()[0])
            late["lag_days_p50_p95_p99"] = list(late_row["p"]) if late_row["p"] else None
            late["col_used"] = lc
        except Exception as e:
            late["error"] = f"{type(e).__name__}: {str(e)[:150]}"
    else:
        late["note"] = "no load-timestamp column; use S2 write cadence as arrival evidence"

    emit("dq_baseline", {
        "basis": "sample (dup check exact on one day)",
        "key_col_null_blank_pct": key_nulls,
        "exclude_hit_x_hit_source_pct": dist,
        "clock_skew": skew,
        "duplicates": dup,
        "late_arrival": late,
    })


run_section("S13", s13_dq_baseline)


>>> running S13 ...
===== BEGIN SHAREABLE: dq_baseline =====
{"basis":"sample (dup check exact on one day)","key_col_null_blank_pct":{"date_time":0.0,"hit_time_gmt":0.0,"visit_num":0.0,"visit_page_num":0.0,"post_event_list":6.893,"pagename":22.201,"page_url":0.001,"exclude_hit":0.0,"hit_source":0.0},"exclude_hit_x_hit_source_pct":[{"exclude_hit":"0","hit_source":"1","pct":94.307},{"exclude_hit":"14","hit_source":"1","pct":4.918},{"exclude_hit":"4","hit_source":"1","pct":0.774},{"exclude_hit":"11","hit_source":"2","pct":0.001}],"clock_skew":{"p5_p50_p95_seconds":[-18000,-14400,-14400],"note":"constant offset = timezone of date_time; spread = clock skew"},"duplicates":{"day":"2026-07-20","rows":37721,"distinct_keys":37630,"dup_pct":0.2412,"key":"post_visid_high,post_visid_low,visit_num,visit_page_num","basis":"exact_one_day"},"late_arrival":{"load_timestamp_cols_found":["load_timestamp"],"lag_days_p50_p95_p99":[1,2,5],"col_used":"load_timestamp"}}
===== END SHAREABLE: dq_baseline =====


## S14 — Identity evidence
Cardinality, null-rate, length, and top-N raw value distributions for every
identity column: validates the CoverMe table's identity story on parity with GWAM's
(`cust_visid` all-null?, `userid` cardinality-1?), and prints the actual values.

In [0]:
def s14_identity():
    ensure_frames()
    identity_cols = [c for c in [
        "mcvisid", "visid_high", "visid_low", "post_visid_high", "post_visid_low",
        "cust_visid", "post_cust_visid", "userid", "username", "user_hash",
        "cookies", "persistent_cookie", "visid_type", "visid_new",
    ] if c in set(DF_S.columns)]

    out = []
    for c in identity_cols:
        r = DF_S.agg(
            F.avg(F.when(nonblank(c), 0.0).otherwise(1.0)).alias("null_blank_frac"),
            F.approx_count_distinct(F.col(c)).alias("apx_distinct"),
            F.min(F.when(nonblank(c), F.length(F.col(c).cast("string")))).alias("len_min"),
            F.avg(F.when(nonblank(c), F.length(F.col(c).cast("string")))).alias("len_avg"),
            F.max(F.when(nonblank(c), F.length(F.col(c).cast("string")))).alias("len_max"),
        ).collect()[0]
        top = (DF_S.filter(nonblank(c)).groupBy(c).count()
                   .orderBy(F.desc("count")).limit(TOP_N).collect())
        n_nonblank = max(SAMPLE_ROWS * (1.0 - (r["null_blank_frac"] or 0.0)), 1)
        out.append({"col": c,
                    "null_blank_pct": round(100.0 * r["null_blank_frac"], 3),
                    "apx_distinct": r["apx_distinct"],
                    "len": {"min": r["len_min"], "avg": r["len_avg"], "max": r["len_max"]},
                    "top": [{"v": str(row[c]),
                             "pct": round(100.0 * row["count"] / n_nonblank, 4)}
                            for row in top]})

    by_col = {o["col"]: o for o in out}
    flags = {
        "cust_visid_all_null": by_col.get("cust_visid", {}).get("null_blank_pct") == 100.0
                               if "cust_visid" in by_col else None,
        "post_cust_visid_all_null": by_col.get("post_cust_visid", {}).get("null_blank_pct") == 100.0
                                    if "post_cust_visid" in by_col else None,
        "userid_cardinality_1": by_col.get("userid", {}).get("apx_distinct") == 1
                                if "userid" in by_col else None,
    }

    ratios = {}
    ts = RESULTS.get("ts_daily", {})
    if ts.get("csv"):
        try:
            rows = [ln.split(",") for ln in ts["csv"]]
            hits = [float(r[1]) for r in rows if len(r) > 3 and r[1]]
            visits = [float(r[2]) for r in rows if len(r) > 3 and r[2]]
            visitors = [float(r[3]) for r in rows if len(r) > 3 and r[3]]
            if visits and visitors:
                ratios = {"mean_hits_per_visit": round(sum(hits) / sum(visits), 3),
                          "mean_visits_per_visitor_daily": round(sum(visits) / sum(visitors), 3)}
        except Exception:
            pass

    emit("identity_evidence", {
        "basis": "sample",
        "note": "raw values printed for every identity column (2026-07-20 authorization)",
        "columns": out, "flags": flags, "daily_ratios": ratios,
    })


run_section("S14", s14_identity)


>>> running S14 ...
===== BEGIN SHAREABLE: identity_evidence =====
{"basis":"sample","note":"raw values printed for every identity column (2026-07-20 authorization)","columns":[{"col":"mcvisid","null_blank_pct":0.0,"apx_distinct":540171,"len":{"min":38,"avg":38.0,"max":38},"top":[{"v":"44434253977085301432663509147490835328","pct":0.056},{"v":"12998207443009767350227282351564661811","pct":0.0313},{"v":"62438021445528036123111449611691583917","pct":0.028},{"v":"06022324086240740240092488218034470138","pct":0.0274},{"v":"08924273519345900981365080869606360580","pct":0.0234},{"v":"12985734394882536454234804598242133925","pct":0.0215},{"v":"18780937595726956330221119255892506704","pct":0.0206},{"v":"52091267921363236258999134996982359573","pct":0.0195},{"v":"87090099860685681231300185543749512110","pct":0.0178},{"v":"36086250934903327683399080367935281935","pct":0.0166},{"v":"00000000000000000000000000000000000000","pct":0.0161},{"v":"52137530871497683808461773341100809436","pct":0.0161},

## S15 — Scope options (three tiered filter profiles)
Reads S3 (rsid inventory), S4 (URL/host inventory), and S11 (geo) to derive three
filter profiles a business user can pick from:
  - **tight**   — single dominant rsid, single primary host, prod-only, top country only.
  - **medium**  — rsids ≥1% share, hosts covering ≥95% of hits, prod+uat, top countries ≥1%.
  - **broad**   — everything except obvious noise (aem_authoring, adobeaemcloud.com).

Each profile carries its own `rsid_list`, `url_include`/`exclude`, `login_host_exclude`,
`geo_country_scope`, `url_col_coalesce_order`, `hit_coverage_pct`, `row_count_estimate`.
The follow-on notebook will paste the picked tier in verbatim as widget defaults.

In [0]:
def s15_scope_options():
    rsid_inv = RESULTS.get("rsid_inventory", {})
    url_inv  = RESULTS.get("url_host_inventory", {})
    geo      = RESULTS.get("geo_language", {})

    if not (rsid_inv and url_inv):
        emit("scope_options", {"error": "rsid_inventory and url_host_inventory required — "
                                        "ensure S3 and S4 ran successfully"})
        return

    per_rsid = rsid_inv.get("per_rsid", []) or []
    hosts    = url_inv.get("s4b_top_hosts", []) or []
    countries = geo.get("top_countries", []) or []
    coalesce_order = url_inv.get("s4a_coalesce_recommended") or []
    total_rows = rsid_inv.get("total_rows_all_history", 0)

    # Single-suite tables have no rsid column; S3 emits a synthetic '<single_suite>'
    # marker. When present, rsid_list stays empty in every tier (no rsid filter to
    # apply) and coverage is computed from URL alone.
    single_suite = bool(rsid_inv.get("single_suite")) or (
        bool(per_rsid) and per_rsid[0].get("rsid") == "<single_suite>"
    )

    # ---- rsid tiers ----
    if single_suite:
        tight_rsids = medium_rsids = broad_rsids = []
    else:
        tight_rsids  = [per_rsid[0]["rsid"]] if per_rsid else []
        medium_rsids = [r["rsid"] for r in per_rsid if r["pct"] >= 1.0]
        broad_rsids  = [r["rsid"] for r in per_rsid]

    # ---- host / URL tiers ----
    prod_hosts = [h for h in hosts if h["env"] == "prod"]
    top_prod_host = prod_hosts[0]["host"] if prod_hosts else (hosts[0]["host"] if hosts else None)

    # Medium: hosts (prod+uat) whose cumulative pct crosses 95%
    medium_url_include, cum = [], 0
    for h in hosts:
        if h["env"] not in ("prod", "uat"):
            continue
        medium_url_include.append(f"%{h['host']}%")
        cum = h["cumulative_pct"]
        if cum >= 95.0:
            break

    broad_exclude = ["%adobeaemcloud.com%"]
    tight_exclude = ["%adobeaemcloud.com%"] + [f"%{h['host']}%" for h in hosts if h["env"] != "prod"][:20]
    medium_exclude = ["%adobeaemcloud.com%"] + [f"%{h['host']}%" for h in hosts if h["env"] == "aem_authoring"]

    # Login-host exclude candidates: anything looking authenticated in the top hosts.
    login_pat = re.compile(r"portal|portail|login|auth|signin|member|sso", re.IGNORECASE)
    login_hosts_seen = [h["host"] for h in hosts if login_pat.search(h["host"] or "")]

    # ---- geo tiers ----
    top_country = countries[0]["country"] if countries else None
    medium_countries = [c["country"] for c in countries if c["pct"] >= 1.0]

    # ---- coverage estimates ----
    # Per-host share = row's cumulative_pct minus the previous row's cumulative_pct.
    def _per_host_share(i, h):
        return h["cumulative_pct"] - (hosts[i - 1]["cumulative_pct"] if i > 0 else 0)

    def _cov_from_hosts(host_list):
        if not host_list or not hosts:
            return None
        return round(sum(_per_host_share(i, h) for i, h in enumerate(hosts)
                         if h["host"] in host_list), 3)

    def _cov_from_rsids(rsid_list):
        # Single-suite: the whole table IS the suite, so coverage is 100% regardless
        # of rsid_list (which is empty by construction above).
        if single_suite:
            return 100.0
        return round(sum(r["pct"] for r in per_rsid if r["rsid"] in rsid_list), 3)

    # Medium host coverage: the last prod-or-uat host's cumulative_pct (the top-N list
    # is already sorted by hits descending, so the last such row IS the coverage number).
    last_medium_host = next((h for h in reversed(hosts) if h["env"] in ("prod", "uat")), None)
    host_medium_cov  = last_medium_host["cumulative_pct"] if last_medium_host else 0.0

    tight_cov  = min(_cov_from_rsids(tight_rsids)  or 0, _cov_from_hosts([top_prod_host]) or 0)
    medium_cov = min(_cov_from_rsids(medium_rsids) or 0, host_medium_cov)
    broad_cov  = 100.0

    def _est(coverage_pct):
        return round(total_rows * coverage_pct / 100.0) if total_rows else None

    scope_option_tight = {
        "label": "tight",
        "description": "Single dominant rsid, single primary prod host, top-1 country only. "
                       "Lowest coverage, highest signal-to-noise.",
        "rsid_list": tight_rsids,
        "url_include": [f"%{top_prod_host}%"] if top_prod_host else [],
        "url_exclude": tight_exclude,
        "login_host_exclude": [f"%{h}%" for h in login_hosts_seen],
        "geo_country_scope": [top_country] if top_country else [],
        "url_col_coalesce_order": coalesce_order,
        "hit_coverage_pct": tight_cov,
        "row_count_estimate": _est(tight_cov),
    }

    scope_option_medium = {
        "label": "medium",
        "description": "rsids ≥1% share, prod+uat hosts covering ≥95%, countries ≥1%.",
        "rsid_list": medium_rsids,
        "url_include": medium_url_include,
        "url_exclude": medium_exclude,
        "login_host_exclude": [f"%{h}%" for h in login_hosts_seen],
        "geo_country_scope": medium_countries,
        "url_col_coalesce_order": coalesce_order,
        "hit_coverage_pct": medium_cov,
        "row_count_estimate": _est(medium_cov),
    }

    scope_option_broad = {
        "label": "broad",
        "description": "All rsids, all hosts; only Adobe AEM authoring excluded. "
                       "Maximum coverage, includes edge cases.",
        "rsid_list": broad_rsids,
        "url_include": [],
        "url_exclude": broad_exclude,
        "login_host_exclude": [],
        "geo_country_scope": [],
        "url_col_coalesce_order": coalesce_order,
        "hit_coverage_pct": broad_cov,
        "row_count_estimate": _est(broad_cov),
    }

    # Unclassified: hosts / prefixes that are non-trivial but sub-1% — the "what should we do
    # with these?" list a business user can act on.
    unclassified = [h for h in hosts if 0.1 <= h["pct"] < 1.0 and h["env"] == "prod"]

    # Mobile-app traffic readout — surfaced from S11b if it ran. Falls back to nulls
    # when the section is absent (older run, or skipped) so downstream stays well-shaped.
    mobile = RESULTS.get("mobile_app_profile") or {}
    top_apps = mobile.get("top_mobile_app_ids") or []
    mobile_traffic = {
        "present": mobile.get("mobile_present"),
        "share_pct": mobile.get("mobile_hit_share_pct"),
        "n_apps_seen": len(top_apps),
        "top_app_id": top_apps[0].get("v") if top_apps else None,
        "recommendation": mobile.get("recommendation"),
    }

    emit("scope_options", {
        "generated_at": datetime.datetime.now().isoformat(timespec="seconds"),
        "single_suite": single_suite,
        "tight":  scope_option_tight,
        "medium": scope_option_medium,
        "broad":  scope_option_broad,
        "unclassified_hosts_prod_0p1_to_1pct": unclassified[:30],
        "mobile_traffic": mobile_traffic,
        "reading_note": (
            "Pick a tier by looking at `hit_coverage_pct` and `row_count_estimate`. "
            "The follow-on coverme_eda.py + coverme_charts.py notebooks paste the "
            "picked tier's fields in as widget defaults. Any host in "
            "`unclassified_hosts_prod_0p1_to_1pct` needs a business decision — "
            "include (medium) or exclude (tight). If `mobile_traffic.present` is "
            "true, add a dedicated mobile-only scope profile alongside the picked tier."
        ),
    })


run_section("S15", s15_scope_options)


>>> running S15 ...
===== BEGIN SHAREABLE: scope_options =====
{"generated_at":"2026-07-22T19:26:34","single_suite":true,"tight":{"label":"tight","description":"Single dominant rsid, single primary prod host, top-1 country only. Lowest coverage, highest signal-to-noise.","rsid_list":[],"url_include":["%coverme.com%"],"url_exclude":["%adobeaemcloud.com%","%uat.coverme.com%","%web-app.uat.affinitylife.aks.manulife.ca%","%author-aem-prod.manulife.ca%","%tripx-web-en.uat.aks.manulife.ca%","%uat.pourmeproteger.com%","%coverme-web.uat.affinityhd.aks.manulife.ca%","%www-aem-stage.coverme.manulife.ca%","%coverme-web.uat.cae.affinityhd.aks.manulife.ca%","%coverme-web.uat.cac.affinityhd.aks.manulife.ca%","%tripx-web-fr.uat.aks.manulife.ca%","%coverme-en.uat.affinityhd.aks.manulife.ca%","%www-aem-stage.coverme.manuvie.ca%","%localhost:5000%","%tripx.uat.coverme.com%"],"login_host_exclude":["%author-aem-prod.manulife.ca%"],"geo_country_scope":["can"],"url_col_coalesce_order":["page_url","visit_st

## S16 — Synthesis spec
Consolidates every prior section into one JSON block optimized for LLM consumption
(building the follow-on `coverme_eda.py` and `coverme_charts.py`). Tolerant of skipped
sections — missing keys just come out as null.

In [0]:
def s16_synthesis_spec():
    expected = ["schema_probe", "delta_meta", "rsid_inventory", "url_host_inventory",
                "daily_volume", "window_frame", "population_census", "live_custom_dims",
                "event_decode", "ts_daily", "ts_events", "ts_profiles", "geo_language",
                "mobile_app_profile", "dim_candidates", "dq_baseline",
                "identity_evidence", "scope_options"]
    missing = [s for s in expected if s not in RESULTS]

    census = RESULTS.get("population_census", {})
    live_dims = {d["col"]: d for d in RESULTS.get("live_custom_dims", {}).get("dims", [])}
    schema_spec = []
    for entry in census.get("populated", []):
        col = entry["col"]
        spec = {"col": col, "dtype": entry.get("dtype"),
                "pop_pct": entry.get("pop_pct"), "apx_distinct": entry.get("apx_distinct")}
        if col in live_dims:
            spec["label"] = live_dims[col].get("label")
            spec["len"] = live_dims[col].get("len")
            spec["top_values"] = live_dims[col].get("top")
        schema_spec.append(spec)

    dv = RESULTS.get("daily_volume", {})
    prof = RESULTS.get("ts_profiles", {})

    emit("synthesis_spec", {
        "meta": {
            "table": TABLE_FQN,
            "generated_at": datetime.datetime.now().isoformat(timespec="seconds"),
            "purpose": "Pre-EDA discovery — outputs feed the tailored coverme_eda.py + coverme_charts.py",
            "scope_mode": "DISCOVER" if not (RSID_LIST or URL_INCLUDE) else "SCOPED",
            "widgets": {"rsid_list": RSID_LIST or None, "url_include": URL_INCLUDE or None,
                        "url_exclude": URL_EXCLUDE or None, "login_host_exclude": LOGIN_EXCLUDE or None,
                        "pdf_labels_path": PDF_LABELS_PATH or None},
            "window": {"start": str(WINDOW_START), "end": str(WINDOW_END),
                       "months": WINDOW_MONTHS},
            "sample_fraction": SAMPLE_FRACTION, "sample_rows": SAMPLE_ROWS,
            "sections_missing": missing, "sections_skipped": SKIPPED,
        },
        "volume": {
            "total_rows_all_history": dv.get("total_rows_all_history"),
            "total_rows_scoped": dv.get("total_rows_scoped"),
            "date_min": dv.get("date_min"), "date_max": dv.get("date_max"),
            "monthly_totals_scoped": dv.get("monthly_totals_scoped"),
            "dow_mean_hits_mon_to_sun": dv.get("dow_mean_scoped_hits_mon_to_sun"),
            "missing_days": dv.get("n_days_missing"),
            "cv": prof.get("cv"),
            "autocorr_lag7": prof.get("autocorr_lag7"),
            "autocorr_lag28": prof.get("autocorr_lag28"),
            "dow_index": prof.get("dow_index_mon_to_sun"),
            "hour_matrix": prof.get("hour_matrix_rows_mon_to_sun_cols_0_23h"),
            "level_shifts": prof.get("level_shift_candidates"),
        },
        "series_ref": "daily values in the ts_daily / ts_events shareable blocks",
        "schema": schema_spec,
        "rsid_inventory": RESULTS.get("rsid_inventory"),
        "url_host_inventory": RESULTS.get("url_host_inventory"),
        "events": RESULTS.get("event_decode"),
        "dims": RESULTS.get("dim_candidates", {}).get("dims"),
        "geo": RESULTS.get("geo_language"),
        "dq": RESULTS.get("dq_baseline"),
        "identity": RESULTS.get("identity_evidence"),
        "mobile": RESULTS.get("mobile_app_profile"),
        "scope_options": RESULTS.get("scope_options"),
    })


run_section("S16", s16_synthesis_spec)


>>> running S16 ...
===== BEGIN SHAREABLE: synthesis_spec =====
----- part 1 of 3 (concatenate parts to reassemble) -----
{"meta":{"table":"csdo_prod_catalog.adobe_coverme_bronze.hit_data","generated_at":"2026-07-22T19:26:34","purpose":"Pre-EDA discovery \u2014 outputs feed the tailored coverme_eda.py + coverme_charts.py","scope_mode":"DISCOVER","widgets":{"rsid_list":null,"url_include":null,"url_exclude":null,"login_host_exclude":null,"pdf_labels_path":null},"window":{"start":"2025-06-01","end":"2026-07-21","months":13},"sample_fraction":0.05,"sample_rows":893283,"sections_missing":[],"sections_skipped":{}},"volume":{"total_rows_all_history":60102247,"total_rows_scoped":60102247,"date_min":"2023-02-28","date_max":"2026-07-21","monthly_totals_scoped":{"2023-02":41717,"2023-03":1099254,"2023-04":894894,"2023-05":930287,"2023-06":943485,"2023-07":923555,"2023-08":916719,"2023-09":808028,"2023-10":1151478,"2023-11":1338573,"2023-12":1696348,"2024-01":2203787,"2024-02":1984131,"2024-03":2

## S17 — Run manifest
Byte length + sha1 of every shareable section, computed from the exact JSON that was
printed. Databricks caps very large per-cell stdout; re-hash any pasted or exported
block and compare here to prove nothing was truncated.

In [0]:
def s17_run_manifest():
    sections = {}
    for sid, payload in RESULTS.items():
        body = json.dumps(payload, separators=(",", ":"), default=str)
        digest = hashlib.sha1(body.encode("utf-8")).hexdigest()
        sections[sid] = {"bytes": len(body),
                         "sha1": "-".join(digest[i:i + 8] for i in range(0, 40, 8))}
    emit("run_manifest", {"sections": sections,
                          "n_sections": len(sections),
                          "skipped": SKIPPED,
                          "note": "dash-grouped sha1 is for readability only; strip '-' to compare."})


run_section("S17", s17_run_manifest)


>>> running S17 ...
===== BEGIN SHAREABLE: run_manifest =====
{"sections":{"pdf_labels":{"bytes":72,"sha1":"7d13be1a-9ea84f68-9c421e4b-db919987-56b980a8"},"schema_probe":{"bytes":31338,"sha1":"d519e815-200aba15-cdb3c6ad-1de795b0-83daee42"},"delta_meta":{"bytes":1538,"sha1":"403ef037-e05365e9-2fe5d692-c369e313-fea87575"},"rsid_inventory":{"bytes":3566,"sha1":"0c047545-92dfe277-6ccfb528-d158c364-c903b805"},"url_host_inventory":{"bytes":21812,"sha1":"225bb92d-09ca2b02-a33483c6-e7fcff76-d7e821ae"},"daily_volume":{"bytes":13069,"sha1":"bc8feaa4-fe43c86b-b7d633f5-aff00da2-c27986ff"},"window_frame":{"bytes":528,"sha1":"35758daa-f106c659-c84b3196-bb1869fe-479f9c9d"},"population_census":{"bytes":15557,"sha1":"f6d1a8f7-41e436dc-5f3e4cc5-41b7b610-24708d5a"},"live_custom_dims":{"bytes":30536,"sha1":"a46a414b-6ee3fadb-a3d16d59-e6b4f83e-c3876092"},"event_decode":{"bytes":13209,"sha1":"95d96f76-af5791c6-7aa2c295-5832feac-8a267b92"},"ts_daily":{"bytes":14070,"sha1":"9220baf1-2cd62acc-0f060e16-44bc710

## Done — how to hand back the results

**Primary: export the run notebook itself.** After a full Run All, use
`File → Export → IPython Notebook (.ipynb)` and drop the file in the repo. Every
`===== BEGIN SHAREABLE: <id> =====` block is captured in the cell outputs, so nothing
has to be copied by hand.

**Verify nothing was truncated.** The final `run_manifest` block lists the byte length
+ sha1 of every section; re-hash an exported block and compare. If a block is cut
mid-JSON, re-run just that section (each `run_section` is independent) or lower the
`max_csv_lines` widget.

**Fallback — copy blocks by hand** (if you can't export), in this priority order:

1. `synthesis_spec` (the master consolidation — if you only send one thing, send this)
2. `scope_options` (the three tiered filter profiles for business-user review)
3. `rsid_inventory` + `url_host_inventory` (the two headline discovery blocks)
4. `mobile_app_profile` (mobile-SDK traffic reading — drives mobile-only scope in the follow-on EDA)
5. `ts_daily` + `ts_events` + `ts_profiles` (daily series + weekly profile for model design)
6. `event_decode` + `live_custom_dims` + `population_census` (metric-registry seeding)
7. `daily_volume` + `delta_meta` + `geo_language` (volume + freshness + geography)
8. `dim_candidates` + `dq_baseline` + `identity_evidence` + `schema_probe` + `run_manifest`

**Scope reminder.** Everything except `delta_meta` and the `total_rows_all_history` field
of `daily_volume` reflects whatever scope the widgets pinned. If every scope widget was
empty (DISCOVER mode, the default), every section covers the whole table. If widgets
were populated, `synthesis_spec.meta.scope_mode` will be `SCOPED` and the numbers apply
only to that subset.

In [0]:
# Release the sample cache so a follow-up cell doesn't inherit stale pinned memory.
if DF_S is not None:
    try:
        DF_S.unpersist()
    except Exception:
        pass